In [1]:
!pip install indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import json
import numpy as np
import re
import string
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import math
from tqdm import tqdm
import csv
from torch.optim.lr_scheduler import ReduceLROnPlateau

nltk.download('punkt')
nltk.download('punkt_tab')

config = {
    "num_epochs": 50,
    "patience": 7,
    "save_best": True,
    "checkpoint_path": "final_best_transformer_en2hi.pt",
    "update_freq": 100,
    "decode_strategy": "beam_search",   # or "greedy", "top_k", "nucleus"
    "bleu_eval_samples": 100,
}


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### **Data Loading from JSON files:**

In [3]:
TRAIN_PATH = '/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json'
VAL_PATH = '/kaggle/input/nlp-capstoneproject/processed/processed/val_data1.json'

with open(TRAIN_PATH, 'r', encoding='utf-8') as file:
    data = json.load(file)

source_sentences_train = []
target_sentences_train = []

source_sentences_val = []
target_sentences_val = []

id_train = []
id_val = []

for language_pair, language_data in data.items():
    if(language_pair == "English-Hindi"):
      print(f"Language Pair: {language_pair}")
      for data_type, data_entries in language_data.items():
          print(f"  Data Type: {data_type}")
          for entry_id, entry_data in data_entries.items():
              source = entry_data["source"]
              target = entry_data["target"]
              if (data_type == "Validation"):
                source_sentences_val.append(source)
                target_sentences_val.append(target)
                id_val.append(entry_id)
              else:
                source_sentences_train.append(source)
                target_sentences_train.append(target)
                id_train.append(entry_id)

with open(VAL_PATH, 'r', encoding='utf-8') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if(language_pair == "English-Hindi"):
      print(f"Language Pair: {language_pair}")
      for data_type, data_entries in language_data.items():
          print(f"  Data Type: {data_type}")
          for entry_id, entry_data in data_entries.items():
              source = entry_data["source"]
              if (data_type == "Validation"):
                source_sentences_val.append(source)
                id_val.append(entry_id)

Language Pair: English-Hindi
  Data Type: Train
Language Pair: English-Hindi
  Data Type: Validation


In [4]:
def preprocess_and_remove_punctuation(sentence):
    sentence = ''.join([char for char in sentence if char not in string.punctuation and not char.isdigit()])
    return sentence

def preprocess_english(sentences):
    tokenized_sentences = [nltk.word_tokenize(preprocess_and_remove_punctuation(sentence.lower())) for sentence in sentences]
    return tokenized_sentences

def preprocess_hindi(sentences):
    normalizer = IndicNormalizerFactory().get_normalizer("hi")
    processed_sentences = []
    for sentence in sentences:
        normalized_sentence = normalizer.normalize(sentence)
        tokens = list(indic_tokenize.trivial_tokenize(normalized_sentence))
        processed_sentences.append(tokens)
    return processed_sentences

# ✅ Step: Reverse source input sentences (improves alignment)
def reverse_sentences(sent_list):
    return [list(reversed(sent)) for sent in sent_list]

target_sentences_train = [re.sub(r'[a-zA-Z]', '', hi) for hi in target_sentences_train]

# After your preprocessing:
english_tokens = preprocess_english(source_sentences_train)
english_test = preprocess_english(source_sentences_val)
hindi_tokens = preprocess_hindi(target_sentences_train)
hindi_test = preprocess_hindi(target_sentences_val)

# 🔁 Apply input reversal trick
en_train = reverse_sentences(english_tokens)
en_test = reverse_sentences(english_test)
de_train = hindi_tokens
de_test = hindi_test

en_index2word = ["<PAD>", "<SOS>", "<EOS>"]
de_index2word = ["<PAD>", "<SOS>", "<EOS>"]
en_vocab = set(en_index2word)
de_vocab = set(de_index2word)
for ds in [en_train, en_test]:
    for sent in ds:
        en_vocab.update(sent)
for ds in [de_train, de_test]:
    for sent in ds:
        de_vocab.update(sent)

en_index2word = list(en_vocab)
de_index2word = list(de_vocab)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
en_word2index = {token: idx for idx, token in enumerate(en_index2word)}
de_word2index = {token: idx for idx, token in enumerate(de_index2word)}


def pad_sequence(seq, vocab, max_len):
    padded_seq = [vocab["<SOS>"]] + [vocab.get(word, vocab["<PAD>"]) for word in seq[:max_len - 2]] + [vocab["<EOS>"]]
    padded_seq += [vocab["<PAD>"]] * (max_len - len(padded_seq))
    return padded_seq[:max_len]

class NMTDataset(Dataset):
    def __init__(self, source, target, source_vocab, target_vocab, max_len):
        self.source = [pad_sequence(sent, source_vocab, max_len) for sent in source]
        self.target = [pad_sequence(sent, target_vocab, max_len) for sent in target]

    def __len__(self):
        return len(self.source)

    def __getitem__(self, idx):
        return torch.tensor(self.source[idx]), torch.tensor(self.target[idx])

BATCH_SIZE = 64
MAX_SEQ_LEN = 25
train_src, val_src, train_tgt, val_tgt = train_test_split(en_train, de_train, test_size=0.1, random_state=42)

train_dataset = NMTDataset(train_src, train_tgt, en_word2index, de_word2index, MAX_SEQ_LEN)
val_dataset = NMTDataset(val_src, val_tgt, en_word2index, de_word2index, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


### **Model Architecture:**

import torch
import torch.nn as nn

# PositionalEncodingLearned (unchanged)
class PositionalEncodingLearned(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.register_buffer("positions", torch.arange(0, max_len).unsqueeze(0))

    def forward(self, x):
        # x expected shape: (batch, seq_len, d_model)
        positions = self.positions[:, :x.size(1)].to(x.device)
        return x + self.pos_embedding(positions)


# Corrected NeuralTransformer
class NeuralTransformer(nn.Module):
    def __init__(self,
                 vocab_size_src,
                 vocab_size_tgt,
                 d_model,
                 nhead,
                 num_enc_layers,
                 num_dec_layers,
                 dropout,
                 max_len,
                 pad_id=0,
                 tie_embeddings=True,
                 pre_norm=True):
        super().__init__()

        # Save pad id for mask creation
        self.pad_id = pad_id
        self.d_model = d_model

        # Embeddings
        self.src_embedding = nn.Embedding(vocab_size_src, d_model, padding_idx=pad_id)
        self.tgt_embedding = nn.Embedding(vocab_size_tgt, d_model, padding_idx=pad_id)

        # Positional embeddings (trainable)
        self.pos_encoding = PositionalEncodingLearned(max_len, d_model)

        # Transformer encoder/decoder layers (pre-norm via norm_first)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True, norm_first=pre_norm
        )
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True, norm_first=pre_norm
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_enc_layers)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_dec_layers)

        # Final normalization layer and output projection
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size_tgt, bias=False)

        # Tie output projection to target embedding weights if desired
        if tie_embeddings:
            try:
                # safe tie: only assign weight reference if dims match
                if self.output_layer.weight.shape == self.tgt_embedding.weight.shape:
                    self.output_layer.weight = self.tgt_embedding.weight
                else:
                    # Shape mismatch -> don't tie
                    print("Warning: embedding tie skipped due to shape mismatch.")
            except Exception as e:
                print("Warning: failed to tie embeddings:", e)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src_indices, tgt_indices):
        """
        src_indices: LongTensor (batch, src_len) token ids (NOT embeddings)
        tgt_indices: LongTensor (batch, tgt_len) token ids (NOT embeddings)
        """

        # --- compute padding masks from index tensors BEFORE embedding ---
        # PyTorch Transformer expects key_padding_mask with shape (batch, seq_len) where True indicates PAD
        src_key_padding_mask = (src_indices == self.pad_id)  # shape: (batch, src_len)
        tgt_key_padding_mask = (tgt_indices == self.pad_id)  # shape: (batch, tgt_len)

        # --- create causal mask for decoder (square subsequent) ---
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_indices.size(1)).to(src_indices.device)

        # --- embed and add positional encodings ---
        src_emb = self.src_embedding(src_indices)  # (batch, src_len, d_model)
        tgt_emb = self.tgt_embedding(tgt_indices)  # (batch, tgt_len, d_model)

        src_emb = self.dropout(self.pos_encoding(src_emb))
        tgt_emb = self.dropout(self.pos_encoding(tgt_emb))

        # --- encode and decode ---
        # encoder: pass src_key_padding_mask to ignore padded positions
        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

        # decoder: pass tgt_mask (causal) and padding masks
        output = self.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        output = self.final_norm(output)  # (batch, tgt_len, d_model)
        logits = self.output_layer(output)  # (batch, tgt_len, vocab_size_tgt)
        return logits


BATCH_SIZE = 64
MAX_SEQ_LEN = 25
EMBED_DIM = 512
NUM_HEADS = 8
DROPOUT = 0.3
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NeuralTransformer(
    vocab_size_src=len(en_word2index),
    vocab_size_tgt=len(de_word2index),
    d_model=EMBED_DIM,
    nhead=NUM_HEADS,
    num_enc_layers=NUM_ENCODER_LAYERS,
    num_dec_layers=NUM_DECODER_LAYERS,
    dropout=DROPOUT,
    max_len=MAX_SEQ_LEN,
    tie_embeddings=True,
    pre_norm=True
).to(device)

print(model)

In [5]:
# This class adds positional information to input using sinusoidal patterns to
# capture sequence order.
class PositionalEncodingSinusoidal(nn.Module):
    def __init__(self, embedding_dimension, maximum_sequence_length):
        super(PositionalEncodingSinusoidal, self).__init__()
        positions = torch.arange(0, maximum_sequence_length).unsqueeze(1).float()
        division_term = torch.exp(torch.arange(0, embedding_dimension, 2).float() * -(math.log(10000.0) / embedding_dimension))
        positional_embeddings = torch.zeros(maximum_sequence_length, embedding_dimension)
        positional_embeddings[:, 0::2] = torch.sin(positions * division_term) #Use sine for even indices (0, 2, 4, ...)
        positional_embeddings[:, 1::2] = torch.cos(positions * division_term) #Using cosine for odd indices (1, 3, 5, ...).
        self.register_buffer('positional_embeddings', positional_embeddings)

    def forward(self, input_tensor):
        sequence_length = input_tensor.size(1)
        return input_tensor + self.positional_embeddings[:sequence_length, :].unsqueeze(0).to(input_tensor.device)

class NeuralTransformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, embedding_dimension, attention_heads, encoder_layers, decoder_layers, dropout_rate, maximum_sequence_length):
        super(NeuralTransformer, self).__init__()
        self.source_embedding = nn.Embedding(source_vocab_size, embedding_dimension)
        self.target_embedding = nn.Embedding(target_vocab_size, embedding_dimension)
        self.positional_encoding = PositionalEncodingSinusoidal(embedding_dimension, maximum_sequence_length)

        self.transformer_core = nn.Transformer(
            d_model=embedding_dimension,
            nhead=attention_heads,
            num_encoder_layers=encoder_layers,
            num_decoder_layers=decoder_layers,
            dropout=dropout_rate,
            batch_first=True
        )
        #Model Output predictions via a linear layer.
        self.output_layer = nn.Linear(embedding_dimension, target_vocab_size)

    def forward(self, source_sequence, target_sequence):
        embedded_source = self.positional_encoding(self.source_embedding(source_sequence))
        embedded_target = self.positional_encoding(self.target_embedding(target_sequence))
        source_mask = self.transformer_core.generate_square_subsequent_mask(source_sequence.size(1)).to(device)
        target_mask = self.transformer_core.generate_square_subsequent_mask(target_sequence.size(1)).to(device)
        transformer_output = self.transformer_core(embedded_source, embedded_target, src_mask=source_mask, tgt_mask=target_mask)
        return self.output_layer(transformer_output)

### **Training and Evaluating the model:**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from torch.optim.lr_scheduler import ReduceLROnPlateau

# -------------------------------
# Translation function
# -------------------------------
import torch
import torch.nn.functional as F
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method4

# -------------------------------------------------------
# Helper: convert indices to words
# -------------------------------------------------------
def idx2words(idx_list, index2word):
    return [index2word[i] for i in idx_list if index2word[i] not in ("<PAD>", "<SOS>", "<EOS>")]


# -------------------------------------------------------
# Helper: sampling from logits (for top-k or nucleus sampling)
# -------------------------------------------------------
def sample_from_logits(logits, top_k=0, top_p=0.0, temperature=1.0):
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)

    # --- Top-k sampling ---
    if top_k > 0:
        top_k = min(top_k, probs.size(-1))
        values, indices = torch.topk(probs, top_k)
        probs = torch.zeros_like(probs).scatter_(-1, indices, values)
        probs = probs / probs.sum()  # renormalize

    # --- Nucleus (top-p) sampling ---
    elif top_p > 0.0:
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cumulative_probs > top_p
        cutoff[..., 1:] = cutoff[..., :-1].clone()
        cutoff[..., 0] = 0
        sorted_probs[cutoff] = 0.0
        probs = torch.zeros_like(probs).scatter_(-1, sorted_indices, sorted_probs)
        probs = probs / probs.sum()  # renormalize

    next_token = torch.multinomial(probs, 1).item()
    return next_token


# -------------------------------------------------------
# Beam search decoding
# -------------------------------------------------------
def beam_search_decode(model, src_tensor, max_len, sos_id, eos_id, beam_size, device):
    """
    Basic beam search for sequence generation.
    """
    model.eval()
    beams = [(torch.tensor([[sos_id]], device=device), 0.0)]  # (sequence, score)

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            with torch.no_grad():
                out = model(src_tensor, seq)
                logits = out[:, -1, :]  # last step
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

            topk_log_probs, topk_ids = torch.topk(log_probs, beam_size)
            for log_p, token_id in zip(topk_log_probs, topk_ids):
                new_seq = torch.cat([seq, token_id.view(1, 1)], dim=1)
                new_score = score + log_p.item()
                new_beams.append((new_seq, new_score))

        # keep best beams
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        # stop if all beams end with EOS
        if all(seq[0, -1].item() == eos_id for seq, _ in beams):
            break

    best_seq = max(beams, key=lambda x: x[1])[0]
    return best_seq.squeeze(0).tolist()


# -------------------------------------------------------
# Sentence translation (supports multiple decoding strategies)
# -------------------------------------------------------
def sentence_translation(
    input_sentence_tokens,
    model,
    src_vocab,
    tgt_index2word,
    max_len,
    device,
    decode_strategy="greedy",
    beam_size=3,
    top_k=50,
    top_p=0.9,
    temperature=1.0,
    tgt_word2index=None,
):
    """
    Autoregressive decoding with: greedy, beam_search, top_k, or nucleus sampling.
    """
    model.eval()
    src_indices = pad_sequence(input_sentence_tokens, src_vocab, max_len)
    src_tensor = torch.tensor(src_indices, dtype=torch.long).unsqueeze(0).to(device)

    # ensure we can access SOS/EOS ids
    if tgt_word2index is None:
        raise ValueError("Must provide tgt_word2index for decoding.")
    sos_id = tgt_word2index["<SOS>"]
    eos_id = tgt_word2index["<EOS>"]

    # --- Beam search ---
    if decode_strategy == "beam_search":
        generated = beam_search_decode(model, src_tensor, max_len, sos_id, eos_id, beam_size, device)
        return idx2words(generated[1:], tgt_index2word)

    # --- Greedy / Sampling-based decoding ---
    generated = [sos_id]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(src_tensor, tgt_tensor)  # [1, tgt_len, vocab]
        logits = out[:, -1, :].squeeze(0)

        if decode_strategy == "greedy":
            next_token = torch.argmax(logits, dim=-1).item()
        elif decode_strategy == "top_k":
            next_token = sample_from_logits(logits, top_k=top_k, temperature=temperature)
        elif decode_strategy == "nucleus":
            next_token = sample_from_logits(logits, top_p=top_p, temperature=temperature)
        else:
            raise ValueError(f"Unknown decode_strategy: {decode_strategy}")

        generated.append(next_token)
        if next_token == eos_id:
            break

    return idx2words(generated[1:], tgt_index2word)


# -------------------------------------------------------
# Validation with BLEU and flexible decoding
# -------------------------------------------------------
def validate_model(
    model,
    val_loader,
    loss_fn,
    src_vocab,
    tgt_vocab,
    tgt_index2word,
    max_len,
    device,
    bleu_samples=200,
    decode_strategy="greedy",
):
    model.eval()
    total_loss = 0.0
    bleu_scores = []
    n_examples = 0

    with torch.no_grad():
        for batch_idx, (src_batch, tgt_batch) in enumerate(val_loader):
            src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
            tgt_inputs = tgt_batch[:, :-1]
            tgt_outputs = tgt_batch[:, 1:]

            preds = model(src_batch, tgt_inputs)
            loss = loss_fn(preds.reshape(-1, preds.shape[-1]), tgt_outputs.reshape(-1))
            total_loss += loss.item()

            batch_size = src_batch.size(0)
            for i in range(batch_size):
                if n_examples >= bleu_samples:
                    break

                # prepare source & reference tokens
                src_tokens = [idx for idx in src_batch[i].cpu().tolist() if idx not in (
                    src_vocab["<PAD>"], src_vocab["<SOS>"], src_vocab["<EOS>"]
                )]
                src_words = [list(src_vocab.keys())[idx] for idx in src_tokens]
                ref_indices = tgt_outputs[i].cpu().tolist()
                ref_tokens = idx2words(ref_indices, tgt_index2word)

                # decode prediction
                pred_tokens = sentence_translation(
                    src_words,
                    model,
                    src_vocab,
                    tgt_index2word,
                    max_len,
                    device,
                    decode_strategy=decode_strategy,
                    tgt_word2index=tgt_vocab,
                )

                if len(ref_tokens) == 0 or len(pred_tokens) == 0:
                    bleu = 0.0
                else:
                    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
                bleu_scores.append(bleu)
                n_examples += 1

    avg_loss = total_loss / len(val_loader)
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    return avg_loss, avg_bleu


# -------------------------------
# Training loop with early stopping
# -------------------------------
import torch
import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.cuda.amp import autocast, GradScaler


def train_transformer_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    loss_fn,
    src_vocab,
    tgt_vocab,
    max_len,
    device,
    config,
    train_src=None,
    tokenizer_en=None,
    tgt_word2index = None
):
    """
    Flexible Transformer training loop with BLEU-based early stopping,
    progress tracking, and multiple decoding strategies.
    """

    # --- Configs ---
    num_epochs = config.get("num_epochs", 20)
    patience = config.get("patience", 5)
    save_best = config.get("save_best", True)
    checkpoint_path = config.get("checkpoint_path", "best_transformer_model.pt")
    update_freq = config.get("update_freq", 100)
    decode_strategy = config.get("decode_strategy", "beam_search")
    bleu_eval_samples = config.get("bleu_eval_samples", 100)

    # --- Initialize tracking variables ---
    scaler = GradScaler()
    best_val_loss = float("inf")
    best_val_bleu = -1.0
    no_improve = 0

    train_losses, val_losses, val_bleus = [], [], []

    # ------------------------------
    # 🚀 Training Loop
    # ------------------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

        # --- Training phase ---
        for batch_idx, (src_batch, tgt_batch) in pbar:
            src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
            tgt_in, tgt_out = tgt_batch[:, :-1], tgt_batch[:, 1:]

            optimizer.zero_grad()
            with autocast():
                preds = model(src_batch, tgt_in)
                loss = loss_fn(preds.reshape(-1, preds.shape[-1]), tgt_out.reshape(-1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            # ✅ Update progress bar occasionally
            if (batch_idx + 1) % update_freq == 0:
                avg_loss_so_far = running_loss / (batch_idx + 1)
                pbar.set_postfix({'avg_loss': f"{avg_loss_so_far:.4f}"})

        # --- End of epoch ---
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- Validation phase ---
        val_loss, val_bleu = validate_model(
            model,
            val_loader,
            loss_fn,
            src_vocab,
            tgt_vocab,
            list(tgt_vocab.keys()),
            max_len,
            device,
            bleu_samples=bleu_eval_samples,
            decode_strategy=decode_strategy,
        )
        val_losses.append(val_loss)
        val_bleus.append(val_bleu)

        scheduler.step(val_loss)

        # --- Epoch summary ---
        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val BLEU: {val_bleu:.4f}")

        # --- Improvement tracking ---
        improved_loss = val_loss < best_val_loss
        improved_bleu = val_bleu > best_val_bleu

        if improved_loss or improved_bleu:
            if improved_loss:
                print(f"✅ Val Loss improved from {best_val_loss:.4f} → {val_loss:.4f}")
                best_val_loss = val_loss
            if improved_bleu:
                print(f"✅ Val BLEU improved from {best_val_bleu:.4f} → {val_bleu:.4f}")
                best_val_bleu = val_bleu
                if save_best:
                    torch.save(model.state_dict(), checkpoint_path)
                    print(f"💾 Saved best model → {checkpoint_path}")
            no_improve = 0
        else:
            no_improve += 1
            print(f"No improvement for {no_improve} epoch(s).")

        # --- Early stopping ---
        if no_improve >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch+1} — "
                  f"no improvement for {patience} epochs.")
            break

        # --- Sanity check translation ---
        if train_src is not None:
            sample_src = random.choice(train_src)
            sample_pred = sentence_translation(
                sample_src,
                model,
                src_vocab,
                list(tgt_vocab.keys()),
                max_len,
                device,
                decode_strategy=decode_strategy,
                tgt_word2index=de_word2index
            )
            try:
                src_text = tokenizer_en.decode(sample_src) if tokenizer_en else " ".join(map(str, sample_src))
                pred_text = " ".join(sample_pred)
            except Exception:
                src_text = " ".join(map(str, sample_src))
                pred_text = " ".join(sample_pred)

            print("\nSample Source:", src_text)
            print("Sample Prediction:", pred_text)
            print("-" * 80)

    # ------------------------------
    # 📈 Plot training curves
    # ------------------------------
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.plot(val_bleus, label="Validation BLEU")
    plt.xlabel("Epochs")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.title("Training & Validation Progress")
    plt.show()

    print(f"\nTraining complete.")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print(f"Best Val BLEU: {best_val_bleu:.4f}")

    return train_losses, val_losses, val_bleus



# -------------------------------
# Setup & Run
# -------------------------------
BATCH_SIZE = 64
MAX_SEQ_LEN = 25
EMBED_DIM = 512
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
DROPOUT = 0.3
LEARNING_RATE = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

'''
model = NeuralTransformer(
    vocab_size_src=len(en_word2index),
    vocab_size_tgt=len(de_word2index),
    d_model=EMBED_DIM,
    nhead=NUM_HEADS,
    num_enc_layers=NUM_ENCODER_LAYERS,
    num_dec_layers=NUM_DECODER_LAYERS,
    dropout=DROPOUT,
    max_len=MAX_SEQ_LEN,
    tie_embeddings=True,
    pre_norm=True
).to(device)'''

model = NeuralTransformer(
    source_vocab_size=len(en_word2index),
    target_vocab_size=len(de_word2index),
    embedding_dimension=EMBED_DIM,
    attention_heads=NUM_HEADS,
    encoder_layers=NUM_ENCODER_LAYERS,
    decoder_layers=NUM_DECODER_LAYERS,
    dropout_rate=DROPOUT,
    maximum_sequence_length=MAX_SEQ_LEN
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=de_word2index["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4,betas=(0.95, 0.99), eps=5e-9)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

train_transformer_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=criterion,
    device=device,
    src_vocab=en_word2index,
    tgt_vocab=de_word2index,
    max_len=MAX_SEQ_LEN,
    config=config,
    train_src=train_src,
    tgt_word2index = de_word2index
)

random_sentence = en_test[0]
translated_sentence = sentence_translation(random_sentence, model, en_word2index, de_word2index, MAX_SEQ_LEN)
print(f"Source: {' '.join(random_sentence)}")
print(f"Translation: {' '.join(translated_sentence)}")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_37/639714112.py:257: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/50:   0%|          | 0/1137 [00:00<?, ?batch/s]

/tmp/ipykernel_37/639714112.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 1/50 Summary:
Train Loss: 6.6194 | Val Loss: 5.9212 | Val BLEU: 0.0193
✅ Val Loss improved from inf → 5.9212
✅ Val BLEU improved from -1.0000 → 0.0193
💾 Saved best model → final_best_transformer_en2hi.pt

Sample Source: management health of method effective an you telling are we here
Sample Prediction: इस प्रकार के लिए , यह एक विशेष रूप से पहले , यह है । । । । ।
--------------------------------------------------------------------------------


Epoch 2/50:   0%|          | 0/1137 [00:00<?, ?batch/s]

...


Epoch 4/50 Summary:
Train Loss: 5.2748 | Val Loss: 5.2360 | Val BLEU: 0.0201
✅ Val Loss improved from 5.3516 → 5.2360

Sample Source: birth take ideas and news the which in situation the observes carefully he definitely but news any in opinion own his put not does he definitely
Sample Prediction: तो , यह एक ( ( । वेरिएबल । ) ) है , यह ( ( । वेरिएबल । ) ) है ,
--------------------------------------------------------------------------------


Epoch 5/50:   0%|          | 0/1137 [00:00<?, ?batch/s]


Epoch 5/50 Summary:
Train Loss: 5.1582 | Val Loss: 5.1686 | Val BLEU: 0.0194
✅ Val Loss improved from 5.2360 → 5.1686

Sample Source: food ladakhi try cuisine kashmiri from change the for
Sample Prediction: क्या मुझे आज रात के बारे में बता सकता है
--------------------------------------------------------------------------------


Epoch 6/50:   0%|          | 0/1137 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [6]:
!pip install indic-nlp-library
!pip uninstall -y torch torchtext
!pip install torch==2.3.1 torchtext==0.17.1 --index-url https://download.pytorch.org/whl/cu121


Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 1.9 MB/s eta 0:00:0000:0100:01
ERROR: Could not find a version that satisfies the requirement torchtext==0.17.1 (from versions: 0.5.0, 0.6.0, 0.15.0+cpu, 0.15.1+cpu, 0.15.2+cpu, 0.16.0+cpu, 0.16.1+cpu, 0.16.2+cpu, 0.17.0+cpu)
ERROR: No matching distribution found for torchtext==0.17.1


In [8]:
# ======================================================================
# EN->BN Transformer with from-scratch BPE tokenizer (no torchtext)
# Features:
#  - BPE tokenizer (trainable on training text)
#  - Reverse source trick
#  - Tied embeddings
#  - Label smoothing optional
#  - Warmup LR scheduler utility
#  - Periodic sample-BLEU printed via tqdm
# ======================================================================

import os, re, json, math, random, time, collections
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from indicnlp.tokenize import indic_tokenize
from sklearn.model_selection import train_test_split
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# 0) SETTINGS
# -------------------------
TRAIN_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json"
MAX_LEN = 30
BATCH_SIZE = 64
N_EPOCHS = 10
BPE_VOCAB_SIZE = 16000     # tune: 8k,16k,32k
MIN_FREQ = 2               # min frequency for BPE tokens to keep
D_MODEL = 256
NHEADS = 8
N_LAYERS = 3
WARMUP_STEPS = 4000
LOG_INTERVAL = 100         # batches
VAL_SAMPLE_BATCHES = 3     # batches for quick BLEU sampling during training

# -------------------------
# 1) DATA LOAD & cleaning
# -------------------------
def clean_text(txt):
    if not isinstance(txt, str):
        return ""
    txt = txt.strip()
    txt = re.sub(r"\s+", " ", txt)
    return txt

def load_raw_pairs(train_json_path, language_pair="English-Bengali", max_len=MAX_LEN):
    with open(train_json_path, "r", encoding="utf-8") as fh:
        data = json.load(fh)
    src_all, tgt_all = [], []
    lp = data.get(language_pair, {})
    for data_type, entries in lp.items():
        for _id, ent in entries.items():
            src = clean_text(ent.get("source", ""))
            tgt = clean_text(ent.get("target", ""))
            if not src or not tgt: continue
            if len(src.split()) <= max_len and len(tgt.split()) <= max_len:
                src_all.append(src.lower())
                tgt_all.append(tgt)  # keep bengali as-is (we'll tokenize with indic)
    return src_all, tgt_all

# -------------------------
# 2) BPE Tokenizer (from scratch)
# Reference: byte-pair encoding style merges on whitespace-word tokens
# -------------------------
def get_vocab_from_corpus(sentences):
    # returns Counter of 'word' -> count, where word starts as characters joined by space + </w>
    vocab = Counter()
    for sent in sentences:
        for word in sent.split():
            # represent word as sequence of chars with end marker
            chars = list(word) + ["</w>"]
            vocab[" ".join(chars)] += 1
    return vocab

def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, v_in):
    v_out = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in v_in.items():
        w_out = word.replace(bigram, replacement)
        v_out[w_out] = freq
    return v_out

def learn_bpe(vocab_counter, num_merges):
    vocab = dict(vocab_counter)
    merges = []
    for i in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best, count = pairs.most_common(1)[0]
        merges.append(best)
        vocab = merge_pair(best, vocab)
    return merges, vocab

def apply_bpe_to_word(word, merges_set):
    # word: string (no spaces)
    symbols = list(word) + ["</w>"]
    # iterative greedy merge using merges_set (which contains tuple pairs)
    i = 0
    while i < len(symbols)-1:
        pair = (symbols[i], symbols[i+1])
        if pair in merges_set:
            symbols[i:i+2] = ["".join(pair)]
            i = max(i-1, 0)
        else:
            i += 1
    # remove end marker join tokens into final tokens
    # combine tokens, remove trailing </w> if present
    tokens = []
    for s in symbols:
        if s.endswith("</w>"):
            tokens.append(s.replace("</w>", ""))
        else:
            tokens.append(s)
    return [t for t in tokens if t != ""]

def build_bpe_tokenizer(sentences, target_vocab_size=BPE_VOCAB_SIZE, min_freq=2):
    vocab_counter = get_vocab_from_corpus(sentences)
    # estimate merges = target_vocab_size - initial_char_types
    # initial vocab size (unique chars + end)
    charset = set()
    for w in vocab_counter:
        for ch in w.split():
            charset.add(ch)
    initial_size = len(charset)
    merges_to_do = max(0, target_vocab_size - initial_size)
    merges, final_vocab = learn_bpe(vocab_counter, merges_to_do)
    merges_set = set(merges)
    # Build token -> id mapping by applying BPE to corpus and counting tokens
    token_freq = Counter()
    for sent in sentences:
        for word in sent.split():
            tokens = apply_bpe_to_word(word, merges_set)
            token_freq.update(tokens)
    # keep tokens with min_freq
    tokens = [t for t, c in token_freq.most_common() if c >= min_freq]
    # add special tokens at front
    special = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
    final_tokens = special + tokens
    token2id = {t:i for i,t in enumerate(final_tokens)}
    id2token = final_tokens
    return {
        "merges": merges,        # ordered list of pairs
        "merges_set": merges_set,
        "token2id": token2id,
        "id2token": id2token
    }

def bpe_encode_sentence(sentence, bpe_obj, lang="en"):
    # for english: whitespace split + bpe on words
    # for bengali: we also split on whitespace, but use indic tokenizer for more accurate tokens if lang='bn'
    toks = sentence.split() if lang=="en" else indic_tokenize.trivial_tokenize(sentence.replace("।"," "), lang="bn")
    out = []
    for w in toks:
        sub = apply_bpe_to_word(w, bpe_obj["merges_set"])
        if len(sub)==0:
            out.append("<UNK>")
        else:
            out.extend([t if t in bpe_obj["token2id"] else "<UNK>" for t in sub])
    return out

def numericalize_tokens(tokens, token2id):
    return [token2id.get(t, token2id["<UNK>"]) for t in tokens]

# -------------------------
# 3) Dataset + collate
# -------------------------
class NMTBPEDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, bpe_src, bpe_tgt, max_len=MAX_LEN, reverse_src=True):
        assert len(src_texts) == len(tgt_texts)
        self.src = src_texts
        self.tgt = tgt_texts
        self.bpe_src = bpe_src
        self.bpe_tgt = bpe_tgt
        self.max_len = max_len
        self.reverse_src = reverse_src

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        s = self.src[idx]
        t = self.tgt[idx]
        if self.reverse_src:
            s = " ".join(list(reversed(s.split())))
        s_toks = bpe_encode_sentence(s, self.bpe_src, lang="en")
        t_toks = bpe_encode_sentence(t, self.bpe_tgt, lang="bn")
        # add SOS/EOS
        t_toks = ["<SOS>"] + t_toks + ["<EOS>"]
        # truncate/pad on collate
        s_ids = numericalize_tokens(s_toks, self.bpe_src["token2id"])
        t_ids = numericalize_tokens(t_toks, self.bpe_tgt["token2id"])
        return torch.tensor(s_ids, dtype=torch.long), torch.tensor(t_ids, dtype=torch.long)

def collate_pad_fn(batch):
    srcs, tgts = zip(*batch)
    # truncate long sequences to MAX_LEN
    srcs = [s[:MAX_LEN] for s in srcs]
    tgts = [t[:MAX_LEN] for t in tgts]
    srcs_p = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=0)
    tgts_p = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=0)
    return srcs_p, tgts_p

# -------------------------
# 4) Transformer model (clean)
# -------------------------
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)
    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        return x + self.pe(positions)

class TransformerNMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx,
                 d_model=D_MODEL, nhead=NHEADS, num_layers=N_LAYERS, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        self.pos = PositionalEmbedding(d_model, max_len=512)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers)
        self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
        # tie embeddings weight
        self.generator.weight = self.tgt_emb.weight

    def forward(self, src, tgt_in):
        src_mask = (src == self.pad_idx)   # [B, Ts]
        tgt_mask = (tgt_in == self.pad_idx)
        src_emb = self.pos(self.src_emb(src))
        tgt_emb = self.pos(self.tgt_emb(tgt_in))
        memory = self.encoder(src_emb, src_key_padding_mask=src_mask)
        seq_len = tgt_in.size(1)
        subsequent = torch.triu(torch.ones((seq_len, seq_len), device=src.device), diagonal=1).bool()
        out = self.decoder(tgt_emb, memory, tgt_mask=subsequent,
                           tgt_key_padding_mask=tgt_mask,
                           memory_key_padding_mask=src_mask)
        logits = self.generator(out)
        return logits

    @torch.no_grad()
    def greedy_decode(self, src, max_len=MAX_LEN, sos_idx=1, eos_idx=2):
        self.eval()
        src_mask = (src == self.pad_idx)
        src_emb = self.pos(self.src_emb(src))
        memory = self.encoder(src_emb, src_key_padding_mask=src_mask)
        ys = torch.full((src.size(0), 1), fill_value=sos_idx, dtype=torch.long, device=src.device)
        for _ in range(max_len-1):
            tgt_emb = self.pos(self.tgt_emb(ys))
            subsequent = torch.triu(torch.ones((ys.size(1), ys.size(1)), device=src.device), diagonal=1).bool()
            out = self.decoder(tgt_emb, memory, tgt_mask=subsequent, memory_key_padding_mask=src_mask)
            next_logits = self.generator(out[:, -1, :])
            next_tokens = next_logits.argmax(-1).unsqueeze(1)
            ys = torch.cat([ys, next_tokens], dim=1)
            if (next_tokens == eos_idx).all():
                break
        return ys

# -------------------------
# 5) Loss & utilities
# -------------------------
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, ignore_index=0):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.ignore_index = ignore_index
    def forward(self, pred, target):
        # pred: [N, C] logits; target: [N]
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(pred, self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
            true_dist.masked_fill_(target.unsqueeze(1) == self.ignore_index, 0.0)
        loss = (-true_dist * pred).sum(dim=1)
        mask = (target != self.ignore_index)
        if mask.any():
            return loss[mask].mean()
        else:
            return loss.mean()

def warmup_lr_lambda(step, d_model=D_MODEL, warmup=WARMUP_STEPS):
    step = max(1, step)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup ** -1.5))

# -------------------------
# 6) BLEU sampling on validation
# -------------------------
def sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=0, max_batches=VAL_SAMPLE_BATCHES):
    model.eval()
    smoothie = SmoothingFunction().method4
    tot, cnt = 0.0, 0
    with torch.no_grad():
        for i, (srcb, tgtb) in enumerate(val_loader):
            if i >= max_batches:
                break
            srcb, tgtb = srcb.to(DEVICE), tgtb.to(DEVICE)
            preds = model.greedy_decode(srcb, max_len=tgtb.size(1))
            for ref, hyp in zip(tgtb, preds):
                # convert
                ref_toks = [id2tok_tgt[idx] for idx in ref.cpu().tolist() if idx not in (0,1,2)]
                hyp_toks = [id2tok_tgt[idx] for idx in hyp.cpu().tolist() if idx not in (0,1,2)]
                if len(ref_toks)==0 or len(hyp_toks)==0:
                    bleu = 0.0
                else:
                    bleu = sentence_bleu([ref_toks], hyp_toks, smoothing_function=smoothie)
                tot += bleu
                cnt += 1
    model.train()
    return tot / max(1, cnt)

# -------------------------
# 7) Orchestrator: prepare data & bpe
# -------------------------
def prepare_all(train_json, val_ratio=0.1, bpe_vocab_size=BPE_VOCAB_SIZE, min_freq=MIN_FREQ):
    src_all, tgt_all = load_raw_pairs(train_json)
    print(f"Loaded total parallel pairs: {len(src_all)}")
    src_train, src_val, tgt_train, tgt_val = train_test_split(src_all, tgt_all, test_size=val_ratio, random_state=42)
    print(f"Train: {len(src_train)}  Val: {len(src_val)}")
    # Build BPEs on train only (separate for en and bn)
    print("Training BPE for English ...")
    bpe_src = build_bpe_tokenizer(src_train, target_vocab_size=bpe_vocab_size, min_freq=min_freq)
    print("Training BPE for Bengali ...")
    bpe_tgt = build_bpe_tokenizer(tgt_train, target_vocab_size=bpe_vocab_size, min_freq=min_freq)
    # prepare datasets
    train_ds = NMTBPEDataset(src_train, tgt_train, bpe_src, bpe_tgt, max_len=MAX_LEN, reverse_src=True)
    val_ds   = NMTBPEDataset(src_val,   tgt_val,   bpe_src, bpe_tgt, max_len=MAX_LEN, reverse_src=True)
    # id2token lists
    id2tok_src = bpe_src["id2token"]
    id2tok_tgt = bpe_tgt["id2token"]
    print(f"Vocab sizes -> src: {len(id2tok_src)}  tgt: {len(id2tok_tgt)}")
    return train_ds, val_ds, bpe_src, bpe_tgt, id2tok_src, id2tok_tgt

# -------------------------
# 8) Main training loop
# -------------------------
def main():
    train_ds, val_ds, bpe_src, bpe_tgt, id2tok_src, id2tok_tgt = prepare_all(TRAIN_JSON, val_ratio=0.1,
                                                                            bpe_vocab_size=BPE_VOCAB_SIZE, min_freq=MIN_FREQ)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pad_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pad_fn, num_workers=2, pin_memory=True)

    src_vocab_size = len(bpe_src["token2id"])
    tgt_vocab_size = len(bpe_tgt["token2id"])
    pad_idx = 0

    model = TransformerNMT(src_vocab_size, tgt_vocab_size, pad_idx, d_model=D_MODEL, nhead=NHEADS, num_layers=N_LAYERS, dim_ff=1024, dropout=0.1).to(DEVICE)
    criterion = LabelSmoothingLoss(classes=tgt_vocab_size, smoothing=0.1, ignore_index=pad_idx)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9,0.98), eps=1e-9)
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: warmup_lr_lambda(step, d_model=D_MODEL, warmup=WARMUP_STEPS))

    best_bleu = 0.0
    global_step = 0

    for epoch in range(1, N_EPOCHS+1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch} [Train]")
        for batch_idx, (src_batch, tgt_batch) in pbar:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            # compute loss and backprop
            tgt_in = tgt_batch[:, :-1]
            tgt_out = tgt_batch[:, 1:]
            logits = model(src_batch, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()  # warmup per-step
            epoch_loss += loss.item()
            global_step += 1
            avg_loss = epoch_loss / (batch_idx + 1)
            pbar.set_postfix({"avg_loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if (batch_idx + 1) % LOG_INTERVAL == 0:
                sample_bleu = sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=pad_idx, max_batches=VAL_SAMPLE_BATCHES)
                print(f"\n🔸 [Epoch {epoch} Batch {batch_idx+1}] sampleBLEU≈{sample_bleu:.4f}  avg_loss≈{avg_loss:.4f}")

        avg_epoch_loss = epoch_loss / len(train_loader)
        val_bleu = sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=pad_idx, max_batches=10)
        print(f"\n[Epoch {epoch}] TrainLoss={avg_epoch_loss:.4f} | ValBLEU={val_bleu:.4f}")

        if val_bleu > best_bleu:
            best_bleu = val_bleu
            torch.save({
                "model_state_dict": model.state_dict(),
                "bpe_src": bpe_src,
                "bpe_tgt": bpe_tgt
            }, "best_transformer_bpe.pth")
            print("✅ New best model saved.")

    print(f"\nTraining finished. Best BLEU: {best_bleu:.4f}")

if __name__ == "__main__":
    main()


Loaded total parallel pairs: 64383
Train: 57944  Val: 6439
Training BPE for English ...


KeyboardInterrupt: 

In [11]:
# ======================================================================
# EN->BN Transformer with custom from-scratch BPE tokenizer (no torchtext)
# ======================================================================

import os, re, json, math, random, time
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from indicnlp.tokenize import indic_tokenize
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 1️⃣ BPE Tokenizer Class
# ----------------------------
from collections import Counter, defaultdict

class BPETokenizer:
    END_WORD = "</w>"

    def __init__(self, vocab_size=8000, min_freq=2):
        self.vocab_size = vocab_size
        self.min_freq = min_freq
        self.merges = []
        self.token2id = {}
        self.id2token = []

    def _word_to_symbols(self, word):
        return tuple(list(word) + [self.END_WORD])

    def _get_pair_stats(self, vocab):
        """Count frequency of symbol pairs in vocab."""
        pair_freq = defaultdict(int)
        for word, freq in vocab.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_freq[pair] += freq
        return pair_freq

    def _merge_vocab(self, pair, vocab):
        """Merge all occurrences of the given pair in the vocab."""
        a, b = pair
        new_vocab = {}
        bigram = re.escape(' '.join(pair))
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        for word, freq in vocab.items():
            word_str = ' '.join(word)
            new_word = pattern.sub(a + b, word_str).split(' ')
            new_vocab[tuple(new_word)] = freq
        return new_vocab

    def fit(self, corpus):
        print(f"🔹 Training BPE (target vocab={self.vocab_size})")

        # Step 1: build vocab on unique words
        word_freq = Counter()
        for sent in corpus:
            for w in sent.split():
                word_freq[w] += 1

        vocab = {self._word_to_symbols(w): f for w, f in word_freq.items()}
        print(f"📘 Unique words: {len(vocab)}")

        # Step 2: main BPE loop
        for i in range(self.vocab_size):
            pairs = self._get_pair_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self._merge_vocab(best, vocab)
            self.merges.append(best)
            if i % 500 == 0 or i == self.vocab_size - 1:
                print(f"   {i:>5}/{self.vocab_size} merges... (pair {best})")

        # Step 3: build final token list
        token_freq = Counter()
        for word, freq in vocab.items():
            for token in word:
                token_freq[token] += freq

        special = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
        kept = [t for t, c in token_freq.items() if c >= self.min_freq]
        vocab_list = special + kept
        self.token2id = {t: i for i, t in enumerate(vocab_list)}
        self.id2token = vocab_list
        print(f"✅ Final vocab size: {len(self.id2token)}")

    def _apply_bpe_word(self, word):
        symbols = list(word) + [self.END_WORD]
        for a, b in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i + 1] == b:
                    symbols[i:i + 2] = [a + b]
                else:
                    i += 1
        return [s for s in symbols if s != self.END_WORD]

    def encode(self, sentence):
        tokens = []
        for w in sentence.split():
            sub_toks = self._apply_bpe_word(w)
            tokens.extend(sub_toks)
        return [self.token2id.get(t, self.token2id["<UNK>"]) for t in tokens]

    def decode(self, ids):
        toks = [self.id2token[i] for i in ids if i < len(self.id2token)]
        toks = [t for t in toks if t not in {"<PAD>", "<SOS>", "<EOS>", "<UNK>"}]
        return " ".join(toks)


# ----------------------------
# 2️⃣ Data Loading & Cleaning
# ----------------------------
def clean_text(txt):
    if not isinstance(txt, str):
        return ""
    txt = re.sub(r"\s+", " ", txt.strip())
    return txt

def load_raw_pairs(json_path, lang_pair="English-Bengali", max_len=30):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    srcs, tgts = [], []
    lp = data[lang_pair]
    for typ, entries in lp.items():
        for _, e in entries.items():
            s, t = clean_text(e["source"]), clean_text(e["target"])
            if s and t and len(s.split()) <= max_len and len(t.split()) <= max_len:
                srcs.append(s.lower())
                tgts.append(t)
    return srcs, tgts

# ----------------------------
# 3️⃣ Dataset
# ----------------------------
class NMTDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, bpe_src, bpe_tgt, max_len=30):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.bpe_src = bpe_src
        self.bpe_tgt = bpe_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_ids = self.bpe_src.encode(self.src_texts[idx], lang="en")
        tgt_ids = self.bpe_tgt.encode(self.tgt_texts[idx], lang="bn")
        tgt_ids = [self.bpe_tgt.token2id["<SOS>"]] + tgt_ids + [self.bpe_tgt.token2id["<EOS>"]]
        return torch.tensor(src_ids[:self.max_len]), torch.tensor(tgt_ids[:self.max_len])

def collate_pad_fn(batch):
    srcs, tgts = zip(*batch)
    srcs_p = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=0)
    tgts_p = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=0)
    return srcs_p, tgts_p

# -------------------------
# 4) Transformer model (clean)
# -------------------------
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)
    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        return x + self.pe(positions)

class TransformerNMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx,
                 d_model=D_MODEL, nhead=NHEADS, num_layers=N_LAYERS, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        self.pos = PositionalEmbedding(d_model, max_len=512)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers)
        self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
        # tie embeddings weight
        self.generator.weight = self.tgt_emb.weight

    def forward(self, src, tgt_in):
        src_mask = (src == self.pad_idx)   # [B, Ts]
        tgt_mask = (tgt_in == self.pad_idx)
        src_emb = self.pos(self.src_emb(src))
        tgt_emb = self.pos(self.tgt_emb(tgt_in))
        memory = self.encoder(src_emb, src_key_padding_mask=src_mask)
        seq_len = tgt_in.size(1)
        subsequent = torch.triu(torch.ones((seq_len, seq_len), device=src.device), diagonal=1).bool()
        out = self.decoder(tgt_emb, memory, tgt_mask=subsequent,
                           tgt_key_padding_mask=tgt_mask,
                           memory_key_padding_mask=src_mask)
        logits = self.generator(out)
        return logits

    @torch.no_grad()
    def greedy_decode(self, src, max_len=MAX_LEN, sos_idx=1, eos_idx=2):
        self.eval()
        src_mask = (src == self.pad_idx)
        src_emb = self.pos(self.src_emb(src))
        memory = self.encoder(src_emb, src_key_padding_mask=src_mask)
        ys = torch.full((src.size(0), 1), fill_value=sos_idx, dtype=torch.long, device=src.device)
        for _ in range(max_len-1):
            tgt_emb = self.pos(self.tgt_emb(ys))
            subsequent = torch.triu(torch.ones((ys.size(1), ys.size(1)), device=src.device), diagonal=1).bool()
            out = self.decoder(tgt_emb, memory, tgt_mask=subsequent, memory_key_padding_mask=src_mask)
            next_logits = self.generator(out[:, -1, :])
            next_tokens = next_logits.argmax(-1).unsqueeze(1)
            ys = torch.cat([ys, next_tokens], dim=1)
            if (next_tokens == eos_idx).all():
                break
        return ys

# -------------------------
# 5) Loss & utilities
# -------------------------
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, ignore_index=0):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.ignore_index = ignore_index
    def forward(self, pred, target):
        # pred: [N, C] logits; target: [N]
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(pred, self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
            true_dist.masked_fill_(target.unsqueeze(1) == self.ignore_index, 0.0)
        loss = (-true_dist * pred).sum(dim=1)
        mask = (target != self.ignore_index)
        if mask.any():
            return loss[mask].mean()
        else:
            return loss.mean()

def warmup_lr_lambda(step, d_model=D_MODEL, warmup=WARMUP_STEPS):
    step = max(1, step)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup ** -1.5))

# -------------------------
# 6) BLEU sampling on validation
# -------------------------
def sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=0, max_batches=VAL_SAMPLE_BATCHES):
    model.eval()
    smoothie = SmoothingFunction().method4
    tot, cnt = 0.0, 0
    with torch.no_grad():
        for i, (srcb, tgtb) in enumerate(val_loader):
            if i >= max_batches:
                break
            srcb, tgtb = srcb.to(DEVICE), tgtb.to(DEVICE)
            preds = model.greedy_decode(srcb, max_len=tgtb.size(1))
            for ref, hyp in zip(tgtb, preds):
                # convert
                ref_toks = [id2tok_tgt[idx] for idx in ref.cpu().tolist() if idx not in (0,1,2)]
                hyp_toks = [id2tok_tgt[idx] for idx in hyp.cpu().tolist() if idx not in (0,1,2)]
                if len(ref_toks)==0 or len(hyp_toks)==0:
                    bleu = 0.0
                else:
                    bleu = sentence_bleu([ref_toks], hyp_toks, smoothing_function=smoothie)
                tot += bleu
                cnt += 1
    model.train()
    return tot / max(1, cnt)

# ----------------------------
# 6️⃣ Data Preparation + Training
# ----------------------------
def prepare_data(train_json, bpe_vocab=8000):
    src, tgt = load_raw_pairs(train_json)
    src_tr, src_val, tgt_tr, tgt_val = train_test_split(src, tgt, test_size=0.1, random_state=42)

    bpe_en = BPETokenizer(vocab_size=bpe_vocab)
    bpe_bn = BPETokenizer(vocab_size=bpe_vocab)
    bpe_en.fit(src_tr)
    bpe_bn.fit(tgt_tr)

    train_ds = NMTDataset(src_tr, tgt_tr, bpe_en, bpe_bn)
    val_ds = NMTDataset(src_val, tgt_val, bpe_en, bpe_bn)
    return train_ds, val_ds, bpe_en, bpe_bn

# ----------------------------
# 7️⃣ Run Main
# ----------------------------
if __name__ == "__main__":
    TRAIN_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json"

    train_ds, val_ds, bpe_en, bpe_bn = prepare_data(TRAIN_JSON, bpe_vocab=8000)
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_pad_fn)
    val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_pad_fn)

    src_vocab_size = len(bpe_src["token2id"])
    tgt_vocab_size = len(bpe_tgt["token2id"])
    pad_idx = 0

    model = TransformerNMT(src_vocab_size, tgt_vocab_size, pad_idx, d_model=D_MODEL, nhead=NHEADS, num_layers=N_LAYERS, dim_ff=1024, dropout=0.1).to(DEVICE)
    criterion = LabelSmoothingLoss(classes=tgt_vocab_size, smoothing=0.1, ignore_index=pad_idx)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9,0.98), eps=1e-9)
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: warmup_lr_lambda(step, d_model=D_MODEL, warmup=WARMUP_STEPS))

    best_bleu = 0.0
    global_step = 0

    for epoch in range(1, N_EPOCHS+1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch} [Train]")
        for batch_idx, (src_batch, tgt_batch) in pbar:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            # compute loss and backprop
            tgt_in = tgt_batch[:, :-1]
            tgt_out = tgt_batch[:, 1:]
            logits = model(src_batch, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()  # warmup per-step
            epoch_loss += loss.item()
            global_step += 1
            avg_loss = epoch_loss / (batch_idx + 1)
            pbar.set_postfix({"avg_loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

            if (batch_idx + 1) % LOG_INTERVAL == 0:
                sample_bleu = sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=pad_idx, max_batches=VAL_SAMPLE_BATCHES)
                print(f"\n🔸 [Epoch {epoch} Batch {batch_idx+1}] sampleBLEU≈{sample_bleu:.4f}  avg_loss≈{avg_loss:.4f}")

        avg_epoch_loss = epoch_loss / len(train_loader)
        val_bleu = sample_bleu_on_val(model, val_loader, id2tok_tgt, pad_idx=pad_idx, max_batches=10)
        print(f"\n[Epoch {epoch}] TrainLoss={avg_epoch_loss:.4f} | ValBLEU={val_bleu:.4f}")

        if val_bleu > best_bleu:
            best_bleu = val_bleu
            torch.save({
                "model_state_dict": model.state_dict(),
                "bpe_src": bpe_src,
                "bpe_tgt": bpe_tgt
            }, "best_transformer_bpe.pth")
            print("✅ New best model saved.")

    print(f"\nTraining finished. Best BLEU: {best_bleu:.4f}")

🔹 Training BPE (target vocab=8000)
📘 Unique words: 70107
       0/8000 merges... (pair ('e', '</w>'))
     500/8000 merges... (pair ('s', 'm'))
    1000/8000 merges... (pair ('he', 'art</w>'))
    1500/8000 merges... (pair ('e', 'e</w>'))
    2000/8000 merges... (pair ('as', '.</w>'))
    2500/8000 merges... (pair ('hous', 'es</w>'))
    3000/8000 merges... (pair ('it', 'ely</w>'))


KeyboardInterrupt: 

In [1]:
!pip install indic-nlp-library torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 23.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: nvidi

In [2]:
# ======================================================================
# 🌐 ENGLISH → BENGALI TRANSFORMER NMT (Stabilized)
# Techniques: Reverse Input, Tied Embeddings, Warmup LR, Label Smoothing
# ======================================================================

import os, re, math, json, random, torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from indicnlp.tokenize import indic_tokenize
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# 1️⃣  LOAD + CLEAN DATA
# ----------------------------------------------------------------------

def clean_text(txt):
    txt = re.sub(r"[^\w\s]", "", txt)
    txt = re.sub(r"\d+", "", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt.lower()

def load_data(language_pair="English-Bengali",
              train_path='/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json',
              split_ratio=0.9,
              max_len=30):
    """
    Loads the English-Bengali dataset from the train JSON, cleans it,
    and splits it into training and validation sets.
    """
    source_sentences, target_sentences, ids = [], [], []

    # -------------------------------
    # Load and parse the JSON
    # -------------------------------
    with open(train_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    print(f"🔹 Loading {language_pair} from {train_path} ...")
    lang_data = data.get(language_pair, {})
    
    for data_type, data_entries in lang_data.items():
        if data_type.lower() in ["train", "training"]:
            for entry_id, entry_data in data_entries.items():
                src = clean_text(entry_data["source"])
                tgt = clean_text(entry_data["target"])
                if len(src.split()) <= max_len and len(tgt.split()) <= max_len:
                    source_sentences.append(src)
                    target_sentences.append(tgt)
                    ids.append(entry_id)

    # -------------------------------
    # Split into Train / Validation
    # -------------------------------
    total = len(source_sentences)
    split_idx = int(total * split_ratio)

    source_sentences_train = source_sentences[:split_idx]
    target_sentences_train = target_sentences[:split_idx]
    id_train = ids[:split_idx]

    source_sentences_val = source_sentences[split_idx:]
    target_sentences_val = target_sentences[split_idx:]
    id_val = ids[split_idx:]

    print(f"\n✅ Data Summary for {language_pair}")
    print(f"  Total: {total} pairs")
    print(f"  Train: {len(source_sentences_train)}")
    print(f"  Val:   {len(source_sentences_val)}\n")

    return (source_sentences_train, target_sentences_train,
            source_sentences_val, target_sentences_val,
            id_train, id_val)

# ----------------------------------------------------------------------
# 2️⃣  TOKENIZATION + VOCAB
# ----------------------------------------------------------------------

def tokenize(sentence, lang="bn"):
    return indic_tokenize.trivial_tokenize(sentence, lang=lang)

# ----------------------------------------------------------------------
# 2️⃣  TEXT CLEANING + VOCAB (PURE PYTORCH)
# ----------------------------------------------------------------------
import string
from collections import Counter

exclude = set(string.punctuation)
remove_digits = str.maketrans('', '', string.digits)

def preprocess_eng_sentence(sent: str) -> str:
    sent = sent.lower()
    sent = sent.replace("'", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    return sent

def preprocess_ban_sentence(sent: str) -> str:
    sent = sent.replace("'", "")
    sent = sent.replace("।", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    sent = "startseq " + sent + " endseq"
    return sent


class Vocab:
    def __init__(self, sentences, min_freq=1, add_special_tokens=True):
        self.freqs = Counter()
        for sent in sentences:
            self.freqs.update(sent.split())

        self.itos = []
        if add_special_tokens:
            self.itos = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]

        for word, freq in self.freqs.items():
            if freq >= min_freq:
                self.itos.append(word)

        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self): return len(self.itos)
    def encode(self, text):
        return [self.stoi.get(tok, self.stoi["<UNK>"]) for tok in text.split()]
    def decode(self, ids):
        return " ".join([self.itos[i] for i in ids if i != self.stoi["<PAD>"]])


def pad_sequences(sequences, max_len, pad_idx):
    padded = []
    for seq in sequences:
        if len(seq) < max_len:
            seq = seq + [pad_idx] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long)


class TranslationDataset(Dataset):
    def __init__(self, en_sentences, bn_sentences, en_vocab, bn_vocab, max_len=30):
        self.en_sentences = en_sentences
        self.bn_sentences = bn_sentences
        self.en_vocab = en_vocab
        self.bn_vocab = bn_vocab
        self.max_len = max_len

    def __len__(self): return len(self.en_sentences)

    def __getitem__(self, idx):
        src_text = self.en_sentences[idx]
        tgt_text = self.bn_sentences[idx]

        src_ids = self.en_vocab.encode(src_text)
        tgt_ids = self.bn_vocab.encode(tgt_text)

        src_ids = pad_sequences([src_ids], self.max_len, self.en_vocab.stoi["<PAD>"])[0]
        tgt_ids = pad_sequences([tgt_ids], self.max_len, self.bn_vocab.stoi["<PAD>"])[0]

        return {"src": src_ids, "tgt": tgt_ids}

def encode(tokens, word2idx, max_len):
    ids = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens]
    ids = ids[:max_len] + [word2idx["<PAD>"]] * (max_len - len(ids))
    return ids

# ----------------------------------------------------------------------
# 3️⃣  DATASET
# ----------------------------------------------------------------------

class NMTDataset(Dataset):
    def __init__(self, src, tgt, src_vocab, tgt_vocab, max_len):
        # 🔹 Ensure both source and target have equal length
        min_len = min(len(src), len(tgt))
        if len(src) != len(tgt):
            print(f"⚠️ Warning: src ({len(src)}) and tgt ({len(tgt)}) lengths differ. "
                  f"Truncating to {min_len}.")
        self.src = src[:min_len]
        self.tgt = tgt[:min_len]
        self.src_vocab, self.tgt_vocab = src_vocab, tgt_vocab
        self.max_len = max_len

    def __len__(self): return len(self.src)

    def __getitem__(self, idx):
        src_toks = list(reversed(self.src[idx].split()))
        tgt_toks = ["<SOS>"] + self.tgt[idx].split() + ["<EOS>"]
    
        # 🔸 Teacher forcing dropout (helps BLEU)
        if random.random() < 0.15:
            rand_idx = random.randint(1, len(tgt_toks) - 2)
            tgt_toks[rand_idx] = "<UNK>"
    
        src_ids = encode(src_toks, self.src_vocab, self.max_len)
        tgt_ids = encode(tgt_toks, self.tgt_vocab, self.max_len)
        return {"src": torch.tensor(src_ids), "tgt": torch.tensor(tgt_ids)}


# ----------------------------------------------------------------------
# 4️⃣  MASKS
# ----------------------------------------------------------------------

def create_padding_mask(seq, pad_idx):
    return (seq == pad_idx).unsqueeze(1).unsqueeze(2)

def create_subsequent_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1).bool()

# ----------------------------------------------------------------------
# 5️⃣  TRANSFORMER MODEL
# ----------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)
    def forward(self, x):
        pos = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        return x + self.pe(pos)

class TransformerNMT(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, pad_idx, d_model=512, nhead=8, num_layers=4, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_embed = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers)
        self.generator = nn.Linear(d_model, tgt_vocab, bias=False)
        # tie weights
        self.generator.weight = self.tgt_embed.weight

    def forward(self, src, tgt):
        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))
        mem = self.encoder(src_emb, src_key_padding_mask=(src==self.pad_idx))
        tgt_mask = create_subsequent_mask(tgt.size(1)).to(tgt.device)
        out = self.decoder(tgt_emb, mem, tgt_mask=tgt_mask,
                           tgt_key_padding_mask=(tgt==self.pad_idx),
                           memory_key_padding_mask=(src==self.pad_idx))
        return self.generator(out)

    @torch.no_grad()
    def greedy_decode(self, src, max_len=30, sos_idx=2, eos_idx=3):
        self.eval()
        src_emb = self.pos_enc(self.src_embed(src))
        mem = self.encoder(src_emb, src_key_padding_mask=(src==self.pad_idx))
        ys = torch.ones(src.size(0), 1, dtype=torch.long, device=src.device) * sos_idx
        for _ in range(max_len-1):
            tgt_emb = self.pos_enc(self.tgt_embed(ys))
            tgt_mask = create_subsequent_mask(ys.size(1)).to(src.device)
            out = self.decoder(tgt_emb, mem, tgt_mask=tgt_mask,
                               memory_key_padding_mask=(src==self.pad_idx))
            next_word = self.generator(out[:, -1]).argmax(-1).unsqueeze(1)
            ys = torch.cat([ys, next_word], dim=1)
            if (next_word == eos_idx).all():
                break
        return ys

# ----------------------------------------------------------------------
# 6️⃣  LABEL SMOOTHING LOSS
# ----------------------------------------------------------------------

class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, ignore_index=0):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.ignore_index = ignore_index
    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        true_dist = torch.zeros_like(pred)
        true_dist.fill_(self.smoothing / (self.cls - 2))
        ignore = target == self.ignore_index
        target = target.masked_fill(ignore, 0)
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist.masked_fill_(ignore.unsqueeze(1), 0)
        loss = (-true_dist * pred).sum(dim=1)
        return loss.mean()

# ----------------------------------------------------------------------
# 7️⃣  TRAIN + EVAL
# ----------------------------------------------------------------------

def train_one_epoch(model, loader, val_loader, optimizer, criterion, pad_idx, epoch, idx2word, log_interval=100):
    model.train()
    total_loss, step = 0, 0
    progress = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch} [Train]", leave=False)

    for i, batch in progress:
        src, tgt = batch["src"].to(device), batch["tgt"].to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

        optimizer.zero_grad()
        logits = model(src, tgt_in)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        step += 1

        # Update tqdm bar with average loss
        avg_loss = total_loss / step
        progress.set_postfix(loss=f"{avg_loss:.3f}")

        # 🔹 Evaluate BLEU every log_interval batches
        if (i + 1) % log_interval == 0:
            model.eval()
            with torch.no_grad():
                sample_bleu = 0
                smoothie = SmoothingFunction().method4
                sample_batches = 0
                for j, val_batch in enumerate(val_loader):
                    if j >= 3:  # only 3 mini-batches (~100 samples)
                        break
                    src_val, tgt_val = val_batch["src"].to(device), val_batch["tgt"].to(device)
                    preds = model.greedy_decode(src_val, max_len=tgt_val.size(1))
                    for ref, hyp in zip(tgt_val, preds):
                        ref_toks = [idx2word[i.item()] for i in ref if i != pad_idx]
                        hyp_toks = [idx2word[i.item()] for i in hyp if i != pad_idx]
                        sample_bleu += sentence_bleu([ref_toks], hyp_toks, smoothing_function=smoothie)
                        sample_batches += 1
                sample_bleu /= max(1, sample_batches)
            model.train()
            print(f"🔸 [Batch {i+1}] AvgLoss={avg_loss:.3f} | SampleBLEU={sample_bleu:.3f}")

    return total_loss / len(loader)

def evaluate_bleu(model, loader, idx2word, pad_idx):
    smoothie = SmoothingFunction().method4
    total_bleu, count = 0, 0
    model.eval()
    with torch.no_grad():
        for batch in loader:
            src, tgt = batch["src"].to(device), batch["tgt"].to(device)
            preds = model.greedy_decode(src, max_len=tgt.size(1))
            for ref, hyp in zip(tgt, preds):
                ref_toks = [idx2word[i.item()] for i in ref if i not in [pad_idx, 2, 3]]
                hyp_toks = [idx2word[i.item()] for i in hyp if i not in [pad_idx, 2, 3]]
                if len(ref_toks) > 2 and len(hyp_toks) > 2:
                    total_bleu += sentence_bleu([ref_toks], hyp_toks, smoothing_function=smoothie)
                    count += 1
    return total_bleu / max(1, count)

# ----------------------------------------------------------------------
# 8️⃣  MAIN TRAINING LOOP
# ----------------------------------------------------------------------

if __name__ == "__main__":
    MAX_LEN, BATCH_SIZE, N_EPOCHS = 30, 64, 30

    train_en, train_bn, val_en, val_bn, id_train, id_val = load_data(max_len=MAX_LEN)

    # 🔹 Preprocess both languages
    train_en = [preprocess_eng_sentence(s) for s in train_en]
    train_bn = [preprocess_ban_sentence(s) for s in train_bn]
    val_en = [preprocess_eng_sentence(s) for s in val_en]
    val_bn = [preprocess_ban_sentence(s) for s in val_bn]
    
    # 🔹 Build vocab using pure PyTorch vocab class
    src_vocab = Vocab(train_en, min_freq=1)
    tgt_vocab = Vocab(train_bn, min_freq=1)
    pad_idx = src_vocab.stoi["<PAD>"]
    
    # 🔹 For BLEU decoding convenience
    src_i2w = src_vocab.itos
    tgt_i2w = tgt_vocab.itos
    
    # 🔹 Create datasets
    train_ds = TranslationDataset(train_en, train_bn, src_vocab, tgt_vocab, MAX_LEN)
    val_ds = TranslationDataset(val_en, val_bn, src_vocab, tgt_vocab, MAX_LEN)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = TransformerNMT(len(src_vocab), len(tgt_vocab), pad_idx).to(device)
    criterion = LabelSmoothingLoss(len(tgt_vocab), smoothing=0.1, ignore_index=pad_idx)
    #criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = optim.Adam(model.parameters(), lr=3e-4, betas=(0.9,0.98), eps=1e-9)
    scheduler = None

    def lr_lambda(step):
        warmup_steps, d_model = 1000, 512
        step = max(1, step)
        return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    #scheduler_warmup = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    #scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

    best_bleu = 0
    for epoch in range(1, N_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_dl, val_dl, optimizer, criterion, pad_idx, epoch, tgt_i2w, log_interval=100)
        val_bleu = evaluate_bleu(model, val_dl, tgt_i2w, pad_idx)
    
        #scheduler_warmup.step()
        #scheduler_plateau.step(val_bleu)
    
        print(f"\n[Epoch {epoch:02}] TrainLoss={train_loss:.3f} | ValBLEU={val_bleu:.3f}")
        if val_bleu > best_bleu:
            best_bleu = val_bleu
            torch.save(model.state_dict(), "best_transformer.pth")
            print("✅ New best model saved.")
    
    print(f"\n🏁 Training complete. Best BLEU = {best_bleu:.3f}")

🔹 Loading English-Bengali from /kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json ...

✅ Data Summary for English-Bengali
  Total: 65347 pairs
  Train: 58812
  Val:   6535



Epoch 1 [Train]:  11%|█         | 99/919 [00:26<03:32,  3.85it/s, loss=25.027]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Epoch 1 [Train]:  11%|█         | 100/919 [00:27<10:33,  1.29it/s, loss=25.027]

🔸 [Batch 100] AvgLoss=25.027 | SampleBLEU=0.013


Epoch 1 [Train]:  22%|██▏       | 200/919 [00:56<09:20,  1.28it/s, loss=20.534]

🔸 [Batch 200] AvgLoss=20.534 | SampleBLEU=0.009


...

🔸 [Batch 900] AvgLoss=10.536 | SampleBLEU=0.011



[Epoch 01] TrainLoss=10.427 | ValBLEU=0.012
✅ New best model saved.


Epoch 2 [Train]:  11%|█         | 100/919 [00:33<12:20,  1.11it/s, loss=5.019]

🔸 [Batch 100] AvgLoss=5.019 | SampleBLEU=0.012


KeyboardInterrupt: 

In [13]:
# train_seq2seq_bahdanau.py
import os
import re
import json
import math
import random
import string
from collections import Counter
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# -------------------------
# Config / Hyperparams
# -------------------------
TRAIN_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LEN = 30             # max sequence length (post padding/truncation)
EMBED_DIM = 512          # embedding dimension
ENC_HIDDEN = 256         # encoder LSTM units per direction -> concat = 512
DEC_HIDDEN = 512         # decoder LSTM hidden size (user requested 512)
BATCH_SIZE = 64
N_EPOCHS = 20
LEARNING_RATE = 1e-3
TEACHER_FORCING_RATIO = 0.5
MIN_FREQ = 1             # min freq for word to be added to vocab
SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# 1) Preprocessing (your logic)
# -------------------------
exclude = set(string.punctuation)
remove_digits = str.maketrans('', '', string.digits)

def preprocess_eng_sentence(sent: str) -> str:
    sent = sent.lower()
    sent = sent.replace("'", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    return sent

def preprocess_ban_sentence(sent: str) -> str:
    sent = sent.replace("'", "")
    sent = sent.replace("।", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    #sent = "startseq " + sent + " endseq"
    return sent

# -------------------------
# 2) Data loading from JSON
# -------------------------
def load_data_from_json(path: str, language_pair="English-Bengali", max_len=MAX_LEN) -> Tuple[List[str], List[str]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    srcs, tgts = [], []
    lang_data = data.get(language_pair, {})
    for dtype, entries in lang_data.items():
        # keep all entries (train/validation keys often present); we'll split after
        for eid, e in entries.items():
            s = e.get("source", "")
            t = e.get("target", "")
            if not s or not t: 
                continue
            s = re.sub(r"[^\w\s\-']", " ", s)  # keep it coarse before preprocess
            t = re.sub(r"[^\w\s\u0980-\u09FF\-।']", " ", t)
            s = s.strip()
            t = t.strip()
            if len(s.split()) <= max_len and len(t.split()) <= max_len:
                srcs.append(s)
                tgts.append(t)
    return srcs, tgts

# -------------------------
# 3) Vocabulary (pure python)
# -------------------------
class Vocab:
    def __init__(self, sentences: List[str], min_freq: int = 1, specials: List[str]=None):
        self.min_freq = min_freq
        self.freqs = Counter()
        for s in sentences:
            self.freqs.update(s.split())
        if specials is None:
            specials = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
        # keep specials in fixed order
        self.itos = list(specials)
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in specials:
                self.itos.append(word)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text: str) -> List[int]:
        return [self.stoi.get(w, self.stoi["<UNK>"]) for w in text.split()]

    def decode(self, ids: List[int]) -> str:
        return " ".join([self.itos[i] for i in ids if i < len(self.itos)])

# -------------------------
# 4) Padding utility (post)
# -------------------------
def pad_sequence_post(seq: List[int], max_len: int, pad_idx: int) -> List[int]:
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [pad_idx] * (max_len - len(seq))

def collate_fn(batch):
    # batch: list of {"src": List[int], "tgt": List[int]}
    srcs = [torch.tensor(x["src"], dtype=torch.long) for x in batch]
    tgts = [torch.tensor(x["tgt"], dtype=torch.long) for x in batch]
    srcs = [s[:MAX_LEN] for s in srcs]
    tgts = [t[:MAX_LEN] for t in tgts]
    srcs_p = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts_p = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs_p, tgts_p

# -------------------------
# 5) Dataset wrapper
# -------------------------
class TranslationDataset(Dataset):
    def __init__(self, src_texts: List[str], tgt_texts: List[str], src_vocab: Vocab, tgt_vocab: Vocab, reverse_src: bool = True):
        assert len(src_texts) == len(tgt_texts)
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.reverse_src = reverse_src

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        s = self.src_texts[idx]
        t = self.tgt_texts[idx]
        if self.reverse_src:
            s = " ".join(list(reversed(s.split())))
        src_ids = self.src_vocab.encode(s)
        tgt_ids = [self.tgt_vocab.stoi["<SOS>"]] + self.tgt_vocab.encode(t) + [self.tgt_vocab.stoi["<EOS>"]]
        src_ids = pad_sequence_post(src_ids, MAX_LEN, PAD_IDX)
        tgt_ids = pad_sequence_post(tgt_ids, MAX_LEN, PAD_IDX)
        return {"src": src_ids, "tgt": tgt_ids}

# -------------------------
# 6) Bahdanau Attention (PyTorch)
# -------------------------
class BahdanauAttention(nn.Module):
    def __init__(self, dec_hidden_dim: int, enc_hidden_dim: int, attn_dim: int = 512):
        """
        dec_hidden_dim: decoder hidden size (512)
        enc_hidden_dim: encoder output dim (bidirectional -> ENC_HIDDEN * 2 = 512)
        """
        super().__init__()
        self.W1 = nn.Linear(dec_hidden_dim, attn_dim, bias=False)   # applied to decoder hidden (state)
        self.W2 = nn.Linear(enc_hidden_dim, attn_dim, bias=False)   # applied to encoder outputs
        self.V  = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, enc_mask: torch.Tensor = None):
        # dec_hidden: [B, dec_hidden_dim]
        # enc_outputs: [B, T_enc, enc_hidden_dim]
        # enc_mask: [B, T_enc] (True for PAD)
        # compute score for each encoder time
        # expand decoder hidden:
        dec_exp = self.W1(dec_hidden).unsqueeze(1)            # [B, 1, attn_dim]
        enc_feat = self.W2(enc_outputs)                       # [B, T, attn_dim]
        score = self.V(torch.tanh(dec_exp + enc_feat)).squeeze(-1)  # [B, T]
        if enc_mask is not None:
            # mask padding positions with large negative
            score = score.masked_fill(enc_mask, -1e9)
        attn_weights = torch.softmax(score, dim=1)            # [B, T]
        # context vector
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)  # [B, enc_hidden_dim]
        return context, attn_weights

# -------------------------
# 7) Encoder / Decoder
# -------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, enc_hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths=None):
        # src: [B, T]
        emb = self.embedding(src)           # [B, T, emb_dim]
        emb = self.dropout(emb)
        # We won't use pack_padded here for simplicity, but mask will be used later
        outputs, (h_n, c_n) = self.lstm(emb)  # outputs: [B, T, 2*enc_hidden]
        # final forward/backward hidden states: h_n shape [num_layers*2, B, enc_hidden]
        # concatenate forward & backward hidden states for initial decoder state
        # take last layer's forward and backward
        forward_h = h_n[-2,:,:]   # [B, enc_hidden]
        backward_h = h_n[-1,:,:]  # [B, enc_hidden]
        dec_h = torch.cat([forward_h, backward_h], dim=1)  # [B, 2*enc_hidden] -> matches DEC_HIDDEN (512)
        forward_c = c_n[-2,:,:]
        backward_c = c_n[-1,:,:]
        dec_c = torch.cat([forward_c, backward_c], dim=1)  # [B, 2*enc_hidden]
        return outputs, (dec_h, dec_c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, dec_hidden, enc_out_dim, attention: BahdanauAttention, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = attention
        # we will input [embedded_token ; context] to LSTM, so input dim = emb_dim + enc_out_dim
        self.lstm = nn.LSTM(emb_dim + enc_out_dim, dec_hidden, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, input_token, prev_hidden, prev_cell, enc_outputs, enc_mask):
        # input_token: [B] (indices)
        emb = self.embedding(input_token).unsqueeze(1)  # [B,1,emb_dim]
        emb = self.dropout(emb)
        dec_hidden = prev_hidden  # [B, dec_hidden] (we pass hidden state)
        # compute attention context using prev_hidden
        context, attn_w = self.attention(prev_hidden, enc_outputs, enc_mask)  # context: [B, enc_out_dim]
        # concat emb and context
        context_exp = context.unsqueeze(1)  # [B,1,enc_out_dim]
        lstm_input = torch.cat([emb, context_exp], dim=-1)  # [B,1, emb+enc_out_dim]
        output, (h_n, c_n) = self.lstm(lstm_input, (prev_hidden.unsqueeze(0), prev_cell.unsqueeze(0)))
        output = output.squeeze(1)  # [B, dec_hidden]
        logits = self.fc_out(output)  # [B, vocab_size]
        # squeezed hidden & cell for next step
        next_hidden = h_n.squeeze(0)  # [B, dec_hidden]
        next_cell = c_n.squeeze(0)    # [B, dec_hidden]
        return logits, next_hidden, next_cell, attn_w

# -------------------------
# 8) Seq2Seq wrapper
# -------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt=None, teacher_forcing_ratio=0.5):
        # src: [B, T_src], tgt: [B, T_tgt] (with SOS..EOS)
        batch_size = src.size(0)
        enc_outputs, (dec_h, dec_c) = self.encoder(src)
        enc_mask = (src == PAD_IDX)  # [B, T_src], True where PAD
        # dec_h, dec_c are [B, dec_hidden]
        max_len = tgt.size(1) if tgt is not None else MAX_LEN
        outputs = torch.zeros(batch_size, max_len, TGT_VOCAB_SIZE).to(DEVICE)
        # first input token is SOS (already in tgt)
        input_token = tgt[:, 0]  # [B]
        hidden = dec_h
        cell = dec_c
        for t in range(1, max_len):
            logits, hidden, cell, attn_w = self.decoder.forward_step(input_token, hidden, cell, enc_outputs, enc_mask)
            outputs[:, t, :] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1)
            if teacher_force and tgt is not None:
                input_token = tgt[:, t]
            else:
                input_token = top1
        return outputs

    def greedy_decode(self, src, max_len=MAX_LEN):
        # returns token ids including initial SOS
        enc_outputs, (dec_h, dec_c) = self.encoder(src)
        enc_mask = (src == PAD_IDX)
        batch_size = src.size(0)
        input_token = torch.full((batch_size,), fill_value=TGT_SOS_IDX, dtype=torch.long, device=src.device)
        hidden = dec_h
        cell = dec_c
        outputs = [input_token.unsqueeze(1)]
        for t in range(max_len-1):
            logits, hidden, cell, attn_w = self.decoder.forward_step(input_token, hidden, cell, enc_outputs, enc_mask)
            next_token = logits.argmax(1)
            outputs.append(next_token.unsqueeze(1))
            input_token = next_token
            if (next_token == TGT_EOS_IDX).all():
                break
        return torch.cat(outputs, dim=1)  # [B, T_out]

# -------------------------
# 9) Loss with masking (SparseCrossEntropy)
# -------------------------
def masked_cross_entropy(logits: torch.Tensor, target: torch.Tensor, pad_idx: int):
    """
    logits: [N, V] or [B, T, V]
    target: [N] or [B, T]
    returns scalar
    """
    if logits.dim() == 3:
        B, T, V = logits.size()
        logits = logits[:, 1:, :].contiguous()   # ignore first timestep (t=0) predictions (no target)
        target = target[:, 1:].contiguous()
        logits = logits.view(-1, V)
        target = target.view(-1)
    else:
        V = logits.size(-1)
    loss = nn.CrossEntropyLoss(reduction="none", ignore_index=pad_idx)
    per_token = loss(logits, target)   # [N] flattened
    mask = (target != pad_idx).float()
    per_token = per_token * mask
    if mask.sum() == 0:
        return per_token.mean()
    return per_token.sum() / mask.sum()

# -------------------------
# 10) Utilities: BLEU sample
# -------------------------
def compute_sample_bleu(model: Seq2Seq, data_loader: DataLoader, ref_vocab: Vocab, max_batches=3):
    smoothie = SmoothingFunction().method4
    model.eval()
    total_bleu, count = 0.0, 0
    with torch.no_grad():
        for i, (src_batch, tgt_batch) in enumerate(data_loader):
            if i >= max_batches:
                break
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            preds = model.greedy_decode(src_batch, max_len=tgt_batch.size(1))
            for ref_ids, pred_ids in zip(tgt_batch.cpu().tolist(), preds.cpu().tolist()):
                # remove PAD and SOS/EOS tokens
                ref_words = [ref_vocab.itos[ix] for ix in ref_ids if ix not in (PAD_IDX, ref_vocab.stoi["<SOS>"], ref_vocab.stoi["<EOS>"])]
                pred_words = [ref_vocab.itos[ix] for ix in pred_ids if ix not in (PAD_IDX, ref_vocab.stoi["<SOS>"], ref_vocab.stoi["<EOS>"])]
                if len(ref_words) == 0 or len(pred_words) == 0:
                    bleu = 0.0
                else:
                    bleu = sentence_bleu([ref_words], pred_words, smoothing_function=smoothie)
                total_bleu += bleu
                count += 1
    model.train()
    return total_bleu / max(1, count)

# -------------------------
# 11) Prepare data + vocabs
# -------------------------
if __name__ == "__main__":
    print("Loading raw data...")
    src_all, tgt_all = load_data_from_json(TRAIN_JSON, max_len=MAX_LEN)
    print(f"Loaded {len(src_all)} pairs")

    # Preprocess
    src_all = [preprocess_eng_sentence(s) for s in src_all]
    tgt_all = [preprocess_ban_sentence(t) for t in tgt_all]

    # split train / val / test using your earlier logic (95% train, next 3% val, last 2% test)
    n = len(src_all)
    train_end = int(0.95 * n)
    val_end = int(0.98 * n)
    train_src = src_all[:train_end]
    train_tgt = tgt_all[:train_end]
    val_src = src_all[train_end:val_end]
    val_tgt = tgt_all[train_end:val_end]
    test_src = src_all[val_end:]
    test_tgt = tgt_all[val_end:]

    print(f"Split sizes -> train: {len(train_src)} | val: {len(val_src)} | test: {len(test_src)}")

    # build vocab only on train
    SRC_SPECIALS = ["<PAD>", "<UNK>"]  # we'll add SOS/EOS to target vocab only
    TGT_SPECIALS = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
    src_vocab = Vocab(train_src, min_freq=MIN_FREQ, specials=SRC_SPECIALS)
    tgt_vocab = Vocab(train_tgt, min_freq=MIN_FREQ, specials=TGT_SPECIALS)

    print(f"Src vocab: {len(src_vocab)} | Tgt vocab: {len(tgt_vocab)}")

    # expose pad and special idxs
    PAD_IDX = src_vocab.stoi["<PAD>"] if "<PAD>" in src_vocab.stoi else 0
    # ensure target special indices exist
    TGT_SOS_IDX = tgt_vocab.stoi["<SOS>"]
    TGT_EOS_IDX = tgt_vocab.stoi["<EOS>"]
    TGT_VOCAB_SIZE = len(tgt_vocab)

    # create datasets & loaders
    train_ds = TranslationDataset(train_src, train_tgt, src_vocab, tgt_vocab, reverse_src=True)
    val_ds = TranslationDataset(val_src, val_tgt, src_vocab, tgt_vocab, reverse_src=True)
    test_ds = TranslationDataset(test_src, test_tgt, src_vocab, tgt_vocab, reverse_src=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

    # -------------------------
    # Build model
    # -------------------------
    enc = Encoder(len(src_vocab), EMBED_DIM, ENC_HIDDEN).to(DEVICE)  # enc_out_dim = ENC_HIDDEN*2 = 512
    attention = BahdanauAttention(dec_hidden_dim=DEC_HIDDEN, enc_hidden_dim=ENC_HIDDEN*2, attn_dim=512).to(DEVICE)
    dec = Decoder(len(tgt_vocab), EMBED_DIM, DEC_HIDDEN, ENC_HIDDEN*2, attention).to(DEVICE)
    model = Seq2Seq(enc, dec).to(DEVICE)

    # optimizer & scheduler
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=True)

    # -------------------------
    # 12) Training with BLEU-based early stopping & translation display
    # -------------------------
    best_val_loss = float("inf")
    best_bleu = 0.0
    no_improve_epochs = 0
    patience = 5
    
    print("Starting training...")
    
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch} [Train]")
        for batch_idx, (src_batch, tgt_batch) in pbar:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
    
            optimizer.zero_grad()
            outputs = model(src_batch, tgt=tgt_batch, teacher_forcing_ratio=TEACHER_FORCING_RATIO)
            loss = masked_cross_entropy(outputs, tgt_batch, PAD_IDX)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    
            epoch_loss += loss.item()
            pbar.set_postfix({"avg_loss": f"{epoch_loss / (batch_idx+1):.4f}"})
    
        avg_epoch_loss = epoch_loss / len(train_loader)
    
        # -------------------
        # Validation phase
        # -------------------
        model.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for src_batch, tgt_batch in val_loader:
                src_batch = src_batch.to(DEVICE)
                tgt_batch = tgt_batch.to(DEVICE)
                outputs = model(src_batch, tgt=tgt_batch, teacher_forcing_ratio=0.0)
                l = masked_cross_entropy(outputs, tgt_batch, PAD_IDX)
                val_loss_total += l.item()
    
        avg_val_loss = val_loss_total / len(val_loader)
        scheduler.step(avg_val_loss)
    
        # -------------------
        # Validation BLEU
        # -------------------
        sample_bleu = compute_sample_bleu(model, val_loader, tgt_vocab, max_batches=3)
    
        print(f"\n[Epoch {epoch}] TrainLoss={avg_epoch_loss:.4f} | ValLoss={avg_val_loss:.4f} | SampleBLEU={sample_bleu:.4f}")
    
        # -------------------
        # Show random validation translation
        # -------------------
        with torch.no_grad():
            src_batch, tgt_batch = next(iter(val_loader))
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            rand_idx = random.randint(0, src_batch.size(0) - 1)
            pred_ids = model.greedy_decode(src_batch[rand_idx:rand_idx+1], max_len=MAX_LEN)
            src_words = [src_vocab.itos[ix] for ix in src_batch[rand_idx].cpu().tolist() if ix != PAD_IDX]
            ref_words = [tgt_vocab.itos[ix] for ix in tgt_batch[rand_idx].cpu().tolist() if ix not in (PAD_IDX, TGT_SOS_IDX, TGT_EOS_IDX)]
            pred_words = [tgt_vocab.itos[ix] for ix in pred_ids.squeeze().cpu().tolist() if ix not in (PAD_IDX, TGT_SOS_IDX, TGT_EOS_IDX)]
    
            print("\n🔹 Random Validation Example:")
            print(f"SRC: {' '.join(src_words)}")
            print(f"REF: {' '.join(ref_words)}")
            print(f"PRED: {' '.join(pred_words)}")
    
        # -------------------
        # Save checkpoints
        # -------------------
        improved = False
    
        # Save if val loss improved
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            improved = True
            ckpt_path = os.path.join(SAVE_DIR, "best_loss_seq2seq_bahdanau.pth")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "src_vocab": src_vocab.itos,
                "tgt_vocab": tgt_vocab.itos
            }, ckpt_path)
            print(f"✅ Saved checkpoint (val loss improved): {ckpt_path}")
    
        # Save if BLEU improved
        if sample_bleu > best_bleu:
            best_bleu = sample_bleu
            improved = True
            ckpt_path = os.path.join(SAVE_DIR, "best_bleu_seq2seq_bahdanau.pth")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "src_vocab": src_vocab.itos,
                "tgt_vocab": tgt_vocab.itos
            }, ckpt_path)
            print(f"✅ Saved checkpoint (BLEU improved): {ckpt_path}")
    
        # -------------------
        # Early stopping check
        # -------------------
        if not improved:
            no_improve_epochs += 1
            print(f"⚠️ No improvement this epoch ({no_improve_epochs}/{patience})")
            if no_improve_epochs >= patience:
                print("🛑 Early stopping triggered due to no BLEU improvement.")
                break
        else:
            no_improve_epochs = 0  # reset patience if improved
    
    # -------------------
    # Final test BLEU
    # -------------------
    print("\nTraining complete. Evaluating on test set...")
    final_bleu = compute_sample_bleu(model, test_loader, tgt_vocab, max_batches=100)
    print(f"🏁 Final Test BLEU (greedy): {final_bleu:.4f}")


Loading raw data...
Loaded 64972 pairs
Split sizes -> train: 61723 | val: 1949 | test: 1300
Src vocab: 48660 | Tgt vocab: 86686
Starting training...


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1 [Train]: 100%|██████████| 965/965 [17:42<00:00,  1.10s/it, avg_loss=7.1571]



[Epoch 1] TrainLoss=7.1571 | ValLoss=7.0398 | SampleBLEU=0.0317

🔹 Random Validation Example:
SRC: kabaddi being other the subcontinent indian the in games tag traditional popular most two the of one is it
REF: startseq এটি ভারতীয় উপমহাদেশের দুটি জনপ্রিয় ঐতিহ্যবাহী ট্যাগ গেমের মধ্যে একটি অন্যটি হল কাবাডি endseq
PRED: startseq এটি একটি একটি একটি একটি একটি একটি একটি একটি একটি একটি একটি একটি একটি endseq endseq
✅ Saved checkpoint (val loss improved): ./checkpoints/best_loss_seq2seq_bahdanau.pth
✅ Saved checkpoint (BLEU improved): ./checkpoints/best_bleu_seq2seq_bahdanau.pth


Epoch 2 [Train]: 100%|██████████| 965/965 [17:43<00:00,  1.10s/it, avg_loss=6.3820]



[Epoch 2] TrainLoss=6.3820 | ValLoss=6.7604 | SampleBLEU=0.0391

🔹 Random Validation Example:
SRC: fossil old years lac for known is jaisalmer near park fossil wood akal
REF: startseq জয়সলমেরের কাছে <UNK> <UNK> <UNK> উদ্যান হল ১৮০ লাখ বছর পুরানো <UNK> জন্য বিখ্যাত ৷ endseq
PRED: startseq বর্তমানে হল হল হল হল হল হল হল বিখ্যাত endseq
✅ Saved checkpoint (val loss improved): ./checkpoints/best_loss_seq2seq_bahdanau.pth
✅ Saved checkpoint (BLEU improved): ./checkpoints/best_bleu_seq2seq_bahdanau.pth


...


[Epoch 18] TrainLoss=2.7223 | ValLoss=7.1930 | SampleBLEU=0.0821

🔹 Random Validation Example:
SRC: area this in planted are insects harmful from themselves save could and soon prepared got dryness resist could which crops the
REF: startseq এই অঞ্চলে বিশেষ করে সেইসব ফসল চাষ করা হয় যেগুলি <UNK> সহ্য করতে পারে শীঘ্রই প্রস্তুত করা যেতে পারে এবং ক্ষতিকারক পোকামাকড় থেকে নিজেদের রক্ষা করতে পারে endseq
PRED: startseq এই যা যা থেকে থেকে থেকে শুরু করে আর আর আর আর আর এই চারা থেকে চারা রোপণ করা যেতে পারে ৷ endseq
⚠️ No improvement this epoch (3/5)


Epoch 19 [Train]: 100%|██████████| 965/965 [17:46<00:00,  1.10s/it, avg_loss=2.6924]



[Epoch 19] TrainLoss=2.6924 | ValLoss=7.2012 | SampleBLEU=0.0818

🔹 Random Validation Example:
SRC: in award achievement lifetime <UNK> with honoured was he
REF: startseq ২০০৯ সালে তিনি সিএনএনআইবিএন লাইফটাইম অ্যাচিভমেন্ট অ্যাওয়ার্ডে ভূষিত হন endseq
PRED: startseq তিনি সালে তিনি কেনটাকি ছবিতে ছবিতে ছবিতে ছবিতেও সম্মানিত হন endseq
⚠️ No improvement this epoch (4/5)


Epoch 20 [Train]: 100%|██████████| 965/965 [17:45<00:00,  1.10s/it, avg_loss=2.6631]



[Epoch 20] TrainLoss=2.6631 | ValLoss=7.2084 | SampleBLEU=0.0816

🔹 Random Validation Example:
SRC: policies socialist towards move to began gradually gandhi elections the following
REF: startseq ১৯৬৭ সালের নির্বাচনের পর গান্ধী ধীরে ধীরে সমাজতান্ত্রিক নীতির দিকে এগোতে শুরু করেন endseq
PRED: startseq শেষ পর্যন্ত গান্ধী বিরোধী সাথে সাথে শুরু করার দিকে শুরু করে endseq
⚠️ No improvement this epoch (5/5)
🛑 Early stopping triggered due to no BLEU improvement.

Training complete. Evaluating on test set...
🏁 Final Test BLEU (greedy): 0.0857


In [14]:
def load_val_data_from_json(path: str, language_pair="English-Bengali", max_len=MAX_LEN) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    srcs = []
    lang_data = data.get(language_pair, {})
    for dtype, entries in lang_data.items():
        for eid, e in entries.items():
            s = e.get("source", "")
            if not s:
                continue
            s = re.sub(r"[^\w\s\-']", " ", s)
            s = s.strip()
            if len(s.split()) <= max_len:
                srcs.append(s)
    return srcs

def predict_sentence(model, en_sentence, src_vocab, tgt_vocab, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        # --- 1️⃣ Preprocess & encode source ---
        s = preprocess_eng_sentence(en_sentence)
        s = " ".join(list(reversed(s.split())))  # if your dataset uses reversed src
        src_ids = src_vocab.encode(s)
        src_ids = pad_sequence_post(src_ids, max_len, src_vocab.stoi["<PAD>"])
        src_tensor = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)

        # --- 2️⃣ Encoder forward ---
        enc_outputs, (dec_h, dec_c) = model.encoder(src_tensor)
        enc_mask = (src_tensor == src_vocab.stoi["<PAD>"])

        # --- 3️⃣ Initialize decoder ---
        input_token = torch.tensor(
            [tgt_vocab.stoi["<SOS>"]], dtype=torch.long, device=DEVICE
        )
        pred_tokens = []
        attn_plots = []

        # --- 4️⃣ Step-by-step decoding ---
        for t in range(max_len):
            logits, dec_h, dec_c, attn_w = model.decoder.forward_step(
                input_token, dec_h, dec_c, enc_outputs, enc_mask
            )
            next_token = logits.argmax(1)  # greedy choice
            attn_plots.append(attn_w.cpu().numpy().reshape(-1))  # store attention weights

            if next_token.item() == tgt_vocab.stoi["<EOS>"]:
                break

            pred_tokens.append(next_token.item())
            input_token = next_token

        # --- 5️⃣ Convert token IDs to words ---
        pred_words = [
            tgt_vocab.itos[ix]
            for ix in pred_tokens
            if ix not in (
                tgt_vocab.stoi["<PAD>"],
                tgt_vocab.stoi["<SOS>"],
                tgt_vocab.stoi["<EOS>"],
            )
        ]
        return " ".join(pred_words), attn_plots

VAL_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/val_data1.json"

val_srcs = load_val_data_from_json(VAL_JSON)

en_vocab, bn_vocab = src_vocab, tgt_vocab

print(f"Total validation samples: {len(val_srcs)}")

# show predictions for first 5 samples
for i in range(5):
    eng = val_srcs[i]
    pred = predict_sentence(model, eng, en_vocab, bn_vocab)
    print(f"\n🔹 SRC: {eng}")
#    print(f"🔸 PRED: {pred}")


Total validation samples: 9315

🔹 SRC: Food parks should be established away from towns and cities in the remote areas so that rural population gets benefits from them and cities do not get overcrowded

🔹 SRC: On one side is the Spiti valley and to the other are numerous C  B   Chandra-Bhaga   range peaks

🔹 SRC: The INF is responsible for compiling world rankings for national teams  maintaining the rules for netball and organising several major international competitions

🔹 SRC: The total population of the district was 192 795 in the 2001 census

🔹 SRC: Patali Srikhetra is a famous place with significant historical importance for Subarnapur district and Odisha  India


In [11]:
val_dataset = TranslationDataset(val_srcs, None, en_vocab, bn_vocab)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate_fn)
bleu = compute_sample_bleu(model, val_loader, bn_vocab, max_batches=len(val_loader))
print(f"Validation BLEU: {bleu:.4f}")

TypeError: object of type 'NoneType' has no len()

In [20]:
def load_val_data_from_json(path: str, language_pair="English-Bengali", max_len=MAX_LEN):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    srcs, ids = [], []
    lang_data = data.get(language_pair, {})

    for dtype, entries in lang_data.items():
        for eid, e in entries.items():
            if data_type == "Validation":
                s = e.get("source", "")
                s = re.sub(r"[^\w\s\-']", " ", s)
                s = s.strip()
                srcs.append(s)
                ids.append(eid)
    return srcs, ids
val_srcs, val_ids = load_val_data_from_json(VAL_JSON)
print(f"Loaded {len(val_srcs)} validation samples.")
import pandas as pd

translations = []

for eid, eng in zip(val_ids, val_srcs):
    pred, _ = predict_sentence(model, eng, en_vocab, bn_vocab)  # returns translation + attn_plot
    translations.append({"ID": eid, "Translation": pred})
df = pd.DataFrame(translations)
output_path = "translations_bengali_seq2seq_greedy.csv"
df.to_csv(output_path, index=False, encoding="utf-8")
print(f"✅ Saved translations to {output_path}")

Loaded 9836 validation samples.
✅ Saved translations to translations_bengali_seq2seq_greedy.csv


In [36]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
import json, re
from tqdm import tqdm

# =========================================================
# 🔧 Helper functions
# =========================================================

def load_val_data_from_json(path: str, language_pair="English-Bengali", max_len=MAX_LEN):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    srcs, ids = [], []
    lang_data = data.get(language_pair, {})

    for dtype, entries in lang_data.items():
        for eid, e in entries.items():
            if dtype == "Validation":
                s = e.get("source", "")
                s = re.sub(r"[^\w\s\-']", " ", s)
                s = s.strip()
                if len(s.split()) <= max_len:
                    srcs.append(s)
                    ids.append(eid)
    return srcs, ids


# =========================================================
# ⚡ Batched prediction function (supports greedy/beam/topk)
# =========================================================

def batch_predict(
    model,
    sentences,
    src_vocab,
    tgt_vocab,
    mode="greedy",
    batch_size=64,
    max_len=MAX_LEN,
    beam_width=3,
    top_k=5,
):
    model.eval()
    all_translations = []

    with torch.no_grad():
        # --- Preprocess and tokenize ---
        encoded_srcs = []
        for s in sentences:
            s = preprocess_eng_sentence(s)
            s = " ".join(list(reversed(s.split())))
            ids = src_vocab.encode(s)
            ids = pad_sequence_post(ids, max_len, src_vocab.stoi["<PAD>"])
            encoded_srcs.append(ids)

        src_tensor = torch.tensor(encoded_srcs, dtype=torch.long, device=DEVICE)
        total_batches = int(np.ceil(len(src_tensor) / batch_size))

        for b in tqdm(range(total_batches), desc=f"Running {mode} decoding", unit="batch"):
            batch_src = src_tensor[b * batch_size : (b + 1) * batch_size]
            batch_size_curr = batch_src.size(0)

            # --- Encoder ---
            enc_outputs, (dec_h, dec_c) = model.encoder(batch_src)
            enc_mask = (batch_src == src_vocab.stoi["<PAD>"])

            # --- Decode according to selected mode ---
            if mode == "greedy":
                preds = _batch_decode_greedy(model, dec_h, dec_c, enc_outputs, enc_mask, tgt_vocab, max_len)
                all_translations.extend(preds)

            elif mode in ["beam", "topk"]:
                preds = []
                # We'll decode each sample in the batch with fast beam / topk:
                for i in range(batch_size_curr):
                    # Handle both 2D and 3D hidden state shapes safely
                    # dec_h may be [num_layers, batch, hidden] or [batch, hidden]
                    if dec_h.dim() == 3:
                        # take the i-th batch column (shape -> [num_layers, 1, hidden])
                        h_i = dec_h[:, i:i+1, :].contiguous()
                        c_i = dec_c[:, i:i+1, :].contiguous()
                    else:
                        # dec_h shape [batch, hidden], make [1, 1, H] for consistency
                        h_i = dec_h[i:i+1, :].unsqueeze(0).contiguous()
                        c_i = dec_c[i:i+1, :].unsqueeze(0).contiguous()

                    enc_i = enc_outputs[i:i+1].contiguous()  # [1, T, H]
                    mask_i = enc_mask[i:i+1].contiguous()    # [1, T]

                    if mode == "beam":
                        pred, _ = _decode_beam_fast(
                            model, h_i, c_i,
                            enc_i, mask_i,
                            tgt_vocab, max_len, beam_width
                        )
                    else:  # topk
                        pred, _ = _decode_topk(
                            model, h_i, c_i,
                            enc_i, mask_i,
                            tgt_vocab, max_len, top_k
                        )
                    preds.append(pred)

                all_translations.extend(preds)

            else:
                raise ValueError("mode must be one of ['greedy','beam','topk']")

    return all_translations


# =========================================================
# 🧩 Decoding methods
# =========================================================

def _batch_decode_greedy(model, dec_h, dec_c, enc_outputs, enc_mask, tgt_vocab, max_len):
    """
    Batched greedy decoding (vectorized across batch).
    This uses whatever dec_h/dec_c shapes your encoder produced.
    """
    batch_size = enc_outputs.size(0)
    input_tokens = torch.full(
        (batch_size,), tgt_vocab.stoi["<SOS>"], dtype=torch.long, device=DEVICE
    )
    preds = torch.zeros((batch_size, max_len), dtype=torch.long, device=DEVICE)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=DEVICE)

    for t in range(max_len):
        logits, dec_h_new, dec_c_new, _ = model.decoder.forward_step(
            input_tokens, (dec_h[-1] if dec_h.dim()==3 else dec_h), (dec_c[-1] if dec_c.dim()==3 else dec_c),
            enc_outputs, enc_mask
        )
        # NOTE: we intentionally update only the top-layer (decoder internal state management may vary)
        # If your decoder returns full structured dec_h/dec_c suitable for next call, you can wire them back.
        next_tokens = logits.argmax(dim=1)
        preds[:, t] = next_tokens
        input_tokens = next_tokens
        finished |= (next_tokens == tgt_vocab.stoi["<EOS>"])
        if finished.all():
            break

    results = []
    for i in range(batch_size):
        pred_ids = preds[i].tolist()
        pred_words = [
            tgt_vocab.itos[ix]
            for ix in pred_ids
            if ix not in (tgt_vocab.stoi["<PAD>"], tgt_vocab.stoi["<SOS>"], tgt_vocab.stoi["<EOS>"])
        ]
        results.append(" ".join(pred_words))
    return results


def _decode_beam_fast(model, dec_h, dec_c, enc_outputs, enc_mask, tgt_vocab, max_len, beam_width=3):
    """
    Optimized beam search decoding (vectorized beams) compatible with attention:
      - enc_outputs: [1, T, H]  -> repeated to [beam, T, H]
      - dec_h, dec_c: expected input as [num_layers, 1, hidden]
    The decoder.attention expects prev_hidden shape [B, hidden] (top layer).
    """
    sos_id = tgt_vocab.stoi["<SOS>"]
    eos_id = tgt_vocab.stoi["<EOS>"]
    device = enc_outputs.device

    # Ensure enc_outputs has batch dim
    if enc_outputs.dim() == 2:
        enc_outputs = enc_outputs.unsqueeze(0)  # [1, T, H]
    B_enc, T, H = enc_outputs.shape[0], enc_outputs.shape[1], enc_outputs.shape[2]

    # Repeat encoder outputs & mask to beam dimension
    enc_outputs_b = enc_outputs.repeat(beam_width, 1, 1)  # [beam, T, H]
    enc_mask_b = enc_mask.repeat(beam_width, 1)           # [beam, T]

    # Extract top-layer hidden/cell and expand to beam_width
    # dec_h: [num_layers, 1, H]  or [1, H]
    if dec_h.dim() == 3:
        dec_h_top = dec_h[-1].squeeze(1)  # [1, H] -> squeeze -> [1,H] or if batch>1 handled earlier; here it's [1,H]
    else:
        dec_h_top = dec_h.squeeze(0)  # ensure [1,H]

    if dec_c.dim() == 3:
        dec_c_top = dec_c[-1].squeeze(1)
    else:
        dec_c_top = dec_c.squeeze(0)

    # Now dec_h_top shape [1, H] -> expand to [beam, H]
    dec_h_top = dec_h_top.repeat(beam_width, 1).to(device)  # [beam, H]
    dec_c_top = dec_c_top.repeat(beam_width, 1).to(device)  # [beam, H]

    # Beam state containers
    input_tokens = torch.full((beam_width,), sos_id, dtype=torch.long, device=device)  # [beam]
    sequences = torch.full((beam_width, 1), sos_id, dtype=torch.long, device=device)   # seq tokens
    beam_scores = torch.zeros(beam_width, device=device)
    finished = torch.zeros(beam_width, dtype=torch.bool, device=device)

    # Iterate and expand beams vectorized
    for _ in range(max_len):
        # forward_step expects prev_hidden as [B, hidden] for attention -- pass dec_h_top
        logits, new_hidden, new_cell, _ = model.decoder.forward_step(
            input_tokens, dec_h_top, dec_c_top, enc_outputs_b, enc_mask_b
        )
        # logits: [beam, vocab]
        log_probs = F.log_softmax(logits, dim=-1)  # [beam, V]
        total_scores = beam_scores.unsqueeze(1) + log_probs  # [beam, V]

        # Flatten beam×vocab and pick topk
        flat_scores = total_scores.view(-1)  # [beam*V]
        topk_scores, topk_indices = torch.topk(flat_scores, k=min(beam_width, flat_scores.size(0)))

        # Convert flat indices back to beam and token indices
        vocab_size = log_probs.size(1)
        beam_idx = (topk_indices // vocab_size)
        token_idx = (topk_indices % vocab_size)

        # Build new sequences and scores
        sequences = torch.cat([sequences[beam_idx], token_idx.unsqueeze(1)], dim=1)  # [beam, seq_len]
        beam_scores = topk_scores

        # Reorder hidden states/new cells according to selected beam_idx
        # new_hidden / new_cell are expected shape [beam, hidden] (as decoder returned)
        # if they are [1, beam, hidden] adjust accordingly
        if new_hidden.dim() == 3:
            # e.g., [num_layers, beam, hidden] -> take top layer
            new_hidden_top = new_hidden[-1]  # [beam, hidden]
        else:
            new_hidden_top = new_hidden  # [beam, hidden]
        if new_cell.dim() == 3:
            new_cell_top = new_cell[-1]
        else:
            new_cell_top = new_cell

        # Reorder per beam indices
        dec_h_top = new_hidden_top[beam_idx]
        dec_c_top = new_cell_top[beam_idx]

        # Next input tokens
        input_tokens = token_idx

        # Mark finished beams where token == EOS
        finished |= token_idx.eq(eos_id)
        if finished.all():
            break

    # pick best sequence
    best_idx = beam_scores.argmax()
    best_seq = sequences[best_idx].tolist()

    pred_words = [
        tgt_vocab.itos[ix]
        for ix in best_seq[1:]
        if ix not in (tgt_vocab.stoi["<PAD>"], tgt_vocab.stoi["<SOS>"], tgt_vocab.stoi["<EOS>"])
    ]
    return " ".join(pred_words), None


def _decode_topk(model, dec_h, dec_c, enc_outputs, enc_mask, tgt_vocab, max_len, top_k=5):
    """
    Per-sample top-k sampling decoding (robust to shape issues).
    """
    device = enc_outputs.device
    sos_id = tgt_vocab.stoi["<SOS>"]
    eos_id = tgt_vocab.stoi["<EOS>"]

    # --- Ensure correct shapes ---
    if enc_outputs.dim() == 2:
        enc_outputs = enc_outputs.unsqueeze(0)  # [1, T, H]
    if enc_mask.dim() == 1:
        enc_mask = enc_mask.unsqueeze(0)        # [1, T]

    # --- Extract top layer hidden ---
    if dec_h.dim() == 3:
        dec_h_top = dec_h[-1].squeeze(0)  # [1, H]
    else:
        dec_h_top = dec_h.squeeze(0)
    if dec_c.dim() == 3:
        dec_c_top = dec_c[-1].squeeze(0)
    else:
        dec_c_top = dec_c.squeeze(0)

    dec_h_top = dec_h_top.unsqueeze(0).to(device)  # [1, H]
    dec_c_top = dec_c_top.unsqueeze(0).to(device)

    input_token = torch.tensor([sos_id], device=device)
    pred_tokens = []

    for _ in range(max_len):
        # 🔧 Ensure tensors stay 3D for attention
        if enc_outputs.dim() == 2:
            enc_outputs = enc_outputs.unsqueeze(0)
        if enc_mask.dim() == 1:
            enc_mask = enc_mask.unsqueeze(0)

        logits, dec_h_new, dec_c_new, _ = model.decoder.forward_step(
            input_token, dec_h_top, dec_c_top, enc_outputs, enc_mask
        )

        probs = F.softmax(logits, dim=-1).squeeze(0)  # [V]
        topk_probs, topk_ids = torch.topk(probs, top_k)
        topk_probs = topk_probs / topk_probs.sum()

        next_token = np.random.choice(topk_ids.cpu().numpy(), p=topk_probs.cpu().numpy())
        if next_token == eos_id:
            break

        pred_tokens.append(int(next_token))
        input_token = torch.tensor([next_token], device=device)

        # Update hidden states
        dec_h_top = (dec_h_new[-1] if dec_h_new.dim() == 3 else dec_h_new).unsqueeze(0)
        dec_c_top = (dec_c_new[-1] if dec_c_new.dim() == 3 else dec_c_new).unsqueeze(0)

    pred_words = [
        tgt_vocab.itos[ix]
        for ix in pred_tokens
        if ix not in (tgt_vocab.stoi["<PAD>"], tgt_vocab.stoi["<SOS>"], tgt_vocab.stoi["<EOS>"])
    ]
    return " ".join(pred_words), None


# =========================================================
# 🚀 Run predictions (example usage)
# =========================================================

VAL_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/val_data1.json"
val_srcs, val_ids = load_val_data_from_json(VAL_JSON)
print(f"Loaded {len(val_srcs)} validation samples.")

# Greedy (batched & fast)
#greedy_preds = batch_predict(model, val_srcs, en_vocab, bn_vocab, mode="greedy", batch_size=128)
#pd.DataFrame({"ID": val_ids, "Translation": greedy_preds}).to_csv("translations_greedy.csv", index=False, encoding="utf-8")

# Beam (vectorized beam per sample)
#beam_preds = batch_predict(model, val_srcs, en_vocab, bn_vocab, mode="beam", batch_size=64, beam_width=5)



Loaded 9315 validation samples.


In [37]:
#pd.DataFrame({"ID": val_ids, "Translation": beam_preds}).to_csv(
#    "translations_bengali_seq2seq_beam.csv", index=False, encoding="utf-8"
#)
#print("✅ Saved beam translations to translations_bengali_seq2seq_beam.csv")

# --- Top-k (per-sample sampling)
topk_preds = batch_predict(model, val_srcs, en_vocab, bn_vocab, mode="topk", batch_size=64, top_k=10)
pd.DataFrame({"ID": val_ids, "Translation": topk_preds}).to_csv(
    "translations_bengali_seq2seq_topk.csv", index=False, encoding="utf-8"
)
print("✅ Saved top-k translations to translations_bengali_seq2seq_topk.csv")

print("✅ Done saving translations.")

Running topk decoding:   0%|          | 0/146 [00:00<?, ?batch/s]


RuntimeError: batch1 must be a 3D tensor

In [21]:
VAL_PATH = '/kaggle/input/nlp-capstoneproject/processed/processed/val_data1.json'

source_sentences_test = []
id_test = []

with open(VAL_PATH, 'r', encoding='utf-8') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if language_pair == "English-Bengali":
        for data_type, data_entries in language_data.items():
            if data_type == "Validation":
                for entry_id, entry_data in data_entries.items():
                    source_sentences_test.append(entry_data["source"])
                    id_test.append(entry_id)

print(f"Total test sentences: {len(source_sentences_test)}")

Total test sentences: 9836


In [4]:
# train_seq2seq_bahdanau.py
import os
import re
import json
import math
import random
import string
from collections import Counter
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# -------------------------
# Config / Hyperparams
# -------------------------
TRAIN_JSON = "/kaggle/input/nlp-capstoneproject/processed/processed/train_data1.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LEN = 50             # max sequence length (post padding/truncation)
EMBED_DIM = 512          # embedding dimension
ENC_HIDDEN = 256         # encoder LSTM units per direction -> concat = 512
DEC_HIDDEN = 512         # decoder LSTM hidden size (user requested 512)
BATCH_SIZE = 64
N_EPOCHS = 20
LEARNING_RATE = 1e-3
TEACHER_FORCING_RATIO = 0.5
MIN_FREQ = 1             # min freq for word to be added to vocab
SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------
# 1) Preprocessing (your logic)
# -------------------------
exclude = set(string.punctuation)
remove_digits = str.maketrans('', '', string.digits)

def preprocess_eng_sentence(sent: str) -> str:
    sent = sent.lower()
    sent = sent.replace("'", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    return sent

def preprocess_ban_sentence(sent: str) -> str:
    sent = sent.replace("'", "")
    sent = sent.replace("।", "")
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent)
    sent = "startseq " + sent + " endseq"
    return sent

# -------------------------
# 2) Data loading from JSON
# -------------------------
def load_data_from_json(path: str, language_pair="English-Bengali", max_len=MAX_LEN) -> Tuple[List[str], List[str]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    srcs, tgts = [], []
    lang_data = data.get(language_pair, {})
    for dtype, entries in lang_data.items():
        # keep all entries (train/validation keys often present); we'll split after
        for eid, e in entries.items():
            s = e.get("source", "")
            t = e.get("target", "")
            if not s or not t: 
                continue
            s = re.sub(r"[^\w\s\-']", " ", s)  # keep it coarse before preprocess
            t = re.sub(r"[^\w\s\-।']", " ", t)
            s = s.strip()
            t = t.strip()
            if len(s.split()) <= max_len and len(t.split()) <= max_len:
                srcs.append(s)
                tgts.append(t)
    return srcs, tgts

# -------------------------
# 3) Vocabulary (pure python)
# -------------------------
class Vocab:
    def __init__(self, sentences: List[str], min_freq: int = 1, specials: List[str]=None):
        self.min_freq = min_freq
        self.freqs = Counter()
        for s in sentences:
            self.freqs.update(s.split())
        if specials is None:
            specials = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
        # keep specials in fixed order
        self.itos = list(specials)
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in specials:
                self.itos.append(word)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text: str) -> List[int]:
        return [self.stoi.get(w, self.stoi["<UNK>"]) for w in text.split()]

    def decode(self, ids: List[int]) -> str:
        return " ".join([self.itos[i] for i in ids if i < len(self.itos)])

# -------------------------
# 4) Padding utility (post)
# -------------------------
def pad_sequence_post(seq: List[int], max_len: int, pad_idx: int) -> List[int]:
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [pad_idx] * (max_len - len(seq))

def collate_fn(batch):
    # batch: list of {"src": List[int], "tgt": List[int]}
    srcs = [torch.tensor(x["src"], dtype=torch.long) for x in batch]
    tgts = [torch.tensor(x["tgt"], dtype=torch.long) for x in batch]
    srcs = [s[:MAX_LEN] for s in srcs]
    tgts = [t[:MAX_LEN] for t in tgts]
    srcs_p = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts_p = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs_p, tgts_p

# -------------------------
# 5) Dataset wrapper
# -------------------------
class TranslationDataset(Dataset):
    def __init__(self, src_texts: List[str], tgt_texts: List[str], src_vocab: Vocab, tgt_vocab: Vocab, reverse_src: bool = True):
        assert len(src_texts) == len(tgt_texts)
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.reverse_src = reverse_src

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        s = self.src_texts[idx]
        t = self.tgt_texts[idx]
        if self.reverse_src:
            s = " ".join(list(reversed(s.split())))
        src_ids = self.src_vocab.encode(s)
        tgt_ids = [self.tgt_vocab.stoi["<SOS>"]] + self.tgt_vocab.encode(t) + [self.tgt_vocab.stoi["<EOS>"]]
        src_ids = pad_sequence_post(src_ids, MAX_LEN, PAD_IDX)
        tgt_ids = pad_sequence_post(tgt_ids, MAX_LEN, PAD_IDX)
        return {"src": src_ids, "tgt": tgt_ids}

# -------------------------
# 6) Bahdanau Attention (PyTorch)
# -------------------------
class BahdanauAttention(nn.Module):
    def __init__(self, dec_hidden_dim: int, enc_hidden_dim: int, attn_dim: int = 512):
        """
        dec_hidden_dim: decoder hidden size (512)
        enc_hidden_dim: encoder output dim (bidirectional -> ENC_HIDDEN * 2 = 512)
        """
        super().__init__()
        self.W1 = nn.Linear(dec_hidden_dim, attn_dim, bias=False)   # applied to decoder hidden (state)
        self.W2 = nn.Linear(enc_hidden_dim, attn_dim, bias=False)   # applied to encoder outputs
        self.V  = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, enc_mask: torch.Tensor = None):
        # dec_hidden: [B, dec_hidden_dim]
        # enc_outputs: [B, T_enc, enc_hidden_dim]
        # enc_mask: [B, T_enc] (True for PAD)
        # compute score for each encoder time
        # expand decoder hidden:
        dec_exp = self.W1(dec_hidden).unsqueeze(1)            # [B, 1, attn_dim]
        enc_feat = self.W2(enc_outputs)                       # [B, T, attn_dim]
        score = self.V(torch.tanh(dec_exp + enc_feat)).squeeze(-1)  # [B, T]
        if enc_mask is not None:
            # mask padding positions with large negative
            score = score.masked_fill(enc_mask, -1e9)
        attn_weights = torch.softmax(score, dim=1)            # [B, T]
        # context vector
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)  # [B, enc_hidden_dim]
        return context, attn_weights

# -------------------------
# 7) Encoder / Decoder
# -------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, enc_hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths=None):
        # src: [B, T]
        emb = self.embedding(src)           # [B, T, emb_dim]
        emb = self.dropout(emb)
        # We won't use pack_padded here for simplicity, but mask will be used later
        outputs, (h_n, c_n) = self.lstm(emb)  # outputs: [B, T, 2*enc_hidden]
        # final forward/backward hidden states: h_n shape [num_layers*2, B, enc_hidden]
        # concatenate forward & backward hidden states for initial decoder state
        # take last layer's forward and backward
        forward_h = h_n[-2,:,:]   # [B, enc_hidden]
        backward_h = h_n[-1,:,:]  # [B, enc_hidden]
        dec_h = torch.cat([forward_h, backward_h], dim=1)  # [B, 2*enc_hidden] -> matches DEC_HIDDEN (512)
        forward_c = c_n[-2,:,:]
        backward_c = c_n[-1,:,:]
        dec_c = torch.cat([forward_c, backward_c], dim=1)  # [B, 2*enc_hidden]
        return outputs, (dec_h, dec_c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, dec_hidden, enc_out_dim, attention: BahdanauAttention, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = attention
        # we will input [embedded_token ; context] to LSTM, so input dim = emb_dim + enc_out_dim
        self.lstm = nn.LSTM(emb_dim + enc_out_dim, dec_hidden, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, input_token, prev_hidden, prev_cell, enc_outputs, enc_mask):
        # input_token: [B] (indices)
        emb = self.embedding(input_token).unsqueeze(1)  # [B,1,emb_dim]
        emb = self.dropout(emb)
        dec_hidden = prev_hidden  # [B, dec_hidden] (we pass hidden state)
        # compute attention context using prev_hidden
        context, attn_w = self.attention(prev_hidden, enc_outputs, enc_mask)  # context: [B, enc_out_dim]
        # concat emb and context
        context_exp = context.unsqueeze(1)  # [B,1,enc_out_dim]
        lstm_input = torch.cat([emb, context_exp], dim=-1)  # [B,1, emb+enc_out_dim]
        output, (h_n, c_n) = self.lstm(lstm_input, (prev_hidden.unsqueeze(0), prev_cell.unsqueeze(0)))
        output = output.squeeze(1)  # [B, dec_hidden]
        logits = self.fc_out(output)  # [B, vocab_size]
        # squeezed hidden & cell for next step
        next_hidden = h_n.squeeze(0)  # [B, dec_hidden]
        next_cell = c_n.squeeze(0)    # [B, dec_hidden]
        return logits, next_hidden, next_cell, attn_w

# -------------------------
# 8) Seq2Seq wrapper
# -------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt=None, teacher_forcing_ratio=0.5):
        # src: [B, T_src], tgt: [B, T_tgt] (with SOS..EOS)
        batch_size = src.size(0)
        enc_outputs, (dec_h, dec_c) = self.encoder(src)
        enc_mask = (src == PAD_IDX)  # [B, T_src], True where PAD
        # dec_h, dec_c are [B, dec_hidden]
        max_len = tgt.size(1) if tgt is not None else MAX_LEN
        outputs = torch.zeros(batch_size, max_len, TGT_VOCAB_SIZE).to(DEVICE)
        # first input token is SOS (already in tgt)
        input_token = tgt[:, 0]  # [B]
        hidden = dec_h
        cell = dec_c
        for t in range(1, max_len):
            logits, hidden, cell, attn_w = self.decoder.forward_step(input_token, hidden, cell, enc_outputs, enc_mask)
            outputs[:, t, :] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1)
            if teacher_force and tgt is not None:
                input_token = tgt[:, t]
            else:
                input_token = top1
        return outputs

    def greedy_decode(self, src, max_len=MAX_LEN):
        # returns token ids including initial SOS
        enc_outputs, (dec_h, dec_c) = self.encoder(src)
        enc_mask = (src == PAD_IDX)
        batch_size = src.size(0)
        input_token = torch.full((batch_size,), fill_value=TGT_SOS_IDX, dtype=torch.long, device=src.device)
        hidden = dec_h
        cell = dec_c
        outputs = [input_token.unsqueeze(1)]
        for t in range(max_len-1):
            logits, hidden, cell, attn_w = self.decoder.forward_step(input_token, hidden, cell, enc_outputs, enc_mask)
            next_token = logits.argmax(1)
            outputs.append(next_token.unsqueeze(1))
            input_token = next_token
            if (next_token == TGT_EOS_IDX).all():
                break
        return torch.cat(outputs, dim=1)  # [B, T_out]

# -------------------------
# 9) Loss with masking (SparseCrossEntropy)
# -------------------------
def masked_cross_entropy(logits: torch.Tensor, target: torch.Tensor, pad_idx: int):
    """
    logits: [N, V] or [B, T, V]
    target: [N] or [B, T]
    returns scalar
    """
    if logits.dim() == 3:
        B, T, V = logits.size()
        logits = logits[:, 1:, :].contiguous()   # ignore first timestep (t=0) predictions (no target)
        target = target[:, 1:].contiguous()
        logits = logits.view(-1, V)
        target = target.view(-1)
    else:
        V = logits.size(-1)
    loss = nn.CrossEntropyLoss(reduction="none", ignore_index=pad_idx)
    per_token = loss(logits, target)   # [N] flattened
    mask = (target != pad_idx).float()
    per_token = per_token * mask
    if mask.sum() == 0:
        return per_token.mean()
    return per_token.sum() / mask.sum()

# -------------------------
# 10) Utilities: BLEU sample
# -------------------------
def compute_sample_bleu(model: Seq2Seq, data_loader: DataLoader, ref_vocab: Vocab, max_batches=3):
    smoothie = SmoothingFunction().method4
    model.eval()
    total_bleu, count = 0.0, 0
    with torch.no_grad():
        for i, (src_batch, tgt_batch) in enumerate(data_loader):
            if i >= max_batches:
                break
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            preds = model.greedy_decode(src_batch, max_len=tgt_batch.size(1))
            for ref_ids, pred_ids in zip(tgt_batch.cpu().tolist(), preds.cpu().tolist()):
                # remove PAD and SOS/EOS tokens
                ref_words = [ref_vocab.itos[ix] for ix in ref_ids if ix not in (PAD_IDX, ref_vocab.stoi["<SOS>"], ref_vocab.stoi["<EOS>"])]
                pred_words = [ref_vocab.itos[ix] for ix in pred_ids if ix not in (PAD_IDX, ref_vocab.stoi["<SOS>"], ref_vocab.stoi["<EOS>"])]
                if len(ref_words) == 0 or len(pred_words) == 0:
                    bleu = 0.0
                else:
                    bleu = sentence_bleu([ref_words], pred_words, smoothing_function=smoothie)
                total_bleu += bleu
                count += 1
    model.train()
    return total_bleu / max(1, count)

# -------------------------
# 11) Prepare data + vocabs
# -------------------------
if __name__ == "__main__":
    print("Loading raw data...")
    src_all, tgt_all = load_data_from_json(TRAIN_JSON, max_len=MAX_LEN)
    print(f"Loaded {len(src_all)} pairs")

    # Preprocess
    src_all = [preprocess_eng_sentence(s) for s in src_all]
    tgt_all = [preprocess_ban_sentence(t) for t in tgt_all]

    # split train / val / test using your earlier logic (95% train, next 3% val, last 2% test)
    n = len(src_all)
    train_end = int(0.95 * n)
    val_end = int(0.98 * n)
    train_src = src_all[:train_end]
    train_tgt = tgt_all[:train_end]
    val_src = src_all[train_end:val_end]
    val_tgt = tgt_all[train_end:val_end]
    test_src = src_all[val_end:]
    test_tgt = tgt_all[val_end:]

    print(f"Split sizes -> train: {len(train_src)} | val: {len(val_src)} | test: {len(test_src)}")

    # build vocab only on train
    SRC_SPECIALS = ["<PAD>", "<UNK>"]  # we'll add SOS/EOS to target vocab only
    TGT_SPECIALS = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
    src_vocab = Vocab(train_src, min_freq=MIN_FREQ, specials=SRC_SPECIALS)
    tgt_vocab = Vocab(train_tgt, min_freq=MIN_FREQ, specials=TGT_SPECIALS)

    print(f"Src vocab: {len(src_vocab)} | Tgt vocab: {len(tgt_vocab)}")

    # expose pad and special idxs
    PAD_IDX = src_vocab.stoi["<PAD>"] if "<PAD>" in src_vocab.stoi else 0
    # ensure target special indices exist
    TGT_SOS_IDX = tgt_vocab.stoi["<SOS>"]
    TGT_EOS_IDX = tgt_vocab.stoi["<EOS>"]
    TGT_VOCAB_SIZE = len(tgt_vocab)

    # create datasets & loaders
    train_ds = TranslationDataset(train_src, train_tgt, src_vocab, tgt_vocab, reverse_src=True)
    val_ds = TranslationDataset(val_src, val_tgt, src_vocab, tgt_vocab, reverse_src=True)
    test_ds = TranslationDataset(test_src, test_tgt, src_vocab, tgt_vocab, reverse_src=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

    # -------------------------
    # Build model
    # -------------------------
    enc = Encoder(len(src_vocab), EMBED_DIM, ENC_HIDDEN).to(DEVICE)  # enc_out_dim = ENC_HIDDEN*2 = 512
    attention = BahdanauAttention(dec_hidden_dim=DEC_HIDDEN, enc_hidden_dim=ENC_HIDDEN*2, attn_dim=512).to(DEVICE)
    dec = Decoder(len(tgt_vocab), EMBED_DIM, DEC_HIDDEN, ENC_HIDDEN*2, attention).to(DEVICE)
    model = Seq2Seq(enc, dec).to(DEVICE)

    # optimizer & scheduler
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=True)

    # -------------------------
    # 12) Training with BLEU-based early stopping & translation display
    # -------------------------
    best_val_loss = float("inf")
    best_bleu = 0.0
    no_improve_epochs = 0
    patience = 5
    
    print("Starting training...")
    
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch} [Train]")
        for batch_idx, (src_batch, tgt_batch) in pbar:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
    
            optimizer.zero_grad()
            outputs = model(src_batch, tgt=tgt_batch, teacher_forcing_ratio=TEACHER_FORCING_RATIO)
            loss = masked_cross_entropy(outputs, tgt_batch, PAD_IDX)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    
            epoch_loss += loss.item()
            pbar.set_postfix({"avg_loss": f"{epoch_loss / (batch_idx+1):.4f}"})
    
        avg_epoch_loss = epoch_loss / len(train_loader)
    
        # -------------------
        # Validation phase
        # -------------------
        model.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for src_batch, tgt_batch in val_loader:
                src_batch = src_batch.to(DEVICE)
                tgt_batch = tgt_batch.to(DEVICE)
                outputs = model(src_batch, tgt=tgt_batch, teacher_forcing_ratio=0.0)
                l = masked_cross_entropy(outputs, tgt_batch, PAD_IDX)
                val_loss_total += l.item()
    
        avg_val_loss = val_loss_total / len(val_loader)
        scheduler.step(avg_val_loss)
    
        # -------------------
        # Validation BLEU
        # -------------------
        sample_bleu = compute_sample_bleu(model, val_loader, tgt_vocab, max_batches=3)
    
        print(f"\n[Epoch {epoch}] TrainLoss={avg_epoch_loss:.4f} | ValLoss={avg_val_loss:.4f} | SampleBLEU={sample_bleu:.4f}")
    
        # -------------------
        # Show random validation translation
        # -------------------
        with torch.no_grad():
            src_batch, tgt_batch = next(iter(val_loader))
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            rand_idx = random.randint(0, src_batch.size(0) - 1)
            pred_ids = model.greedy_decode(src_batch[rand_idx:rand_idx+1], max_len=MAX_LEN)
            src_words = [src_vocab.itos[ix] for ix in src_batch[rand_idx].cpu().tolist() if ix != PAD_IDX]
            ref_words = [tgt_vocab.itos[ix] for ix in tgt_batch[rand_idx].cpu().tolist() if ix not in (PAD_IDX, TGT_SOS_IDX, TGT_EOS_IDX)]
            pred_words = [tgt_vocab.itos[ix] for ix in pred_ids.squeeze().cpu().tolist() if ix not in (PAD_IDX, TGT_SOS_IDX, TGT_EOS_IDX)]
    
            print("\n🔹 Random Validation Example:")
            print(f"SRC: {' '.join(src_words)}")
            print(f"REF: {' '.join(ref_words)}")
            print(f"PRED: {' '.join(pred_words)}")
    
        # -------------------
        # Save checkpoints
        # -------------------
        improved = False
    
        # Save if val loss improved
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            improved = True
            ckpt_path = os.path.join(SAVE_DIR, "best_loss_seq2seq_bahdanau.pth")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "src_vocab": src_vocab.itos,
                "tgt_vocab": tgt_vocab.itos
            }, ckpt_path)
            print(f"✅ Saved checkpoint (val loss improved): {ckpt_path}")
    
        # Save if BLEU improved
        if sample_bleu > best_bleu:
            best_bleu = sample_bleu
            improved = True
            ckpt_path = os.path.join(SAVE_DIR, "best_bleu_seq2seq_bahdanau.pth")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "src_vocab": src_vocab.itos,
                "tgt_vocab": tgt_vocab.itos
            }, ckpt_path)
            print(f"✅ Saved checkpoint (BLEU improved): {ckpt_path}")
    
        # -------------------
        # Early stopping check
        # -------------------
        if not improved:
            no_improve_epochs += 1
            print(f"⚠️ No improvement this epoch ({no_improve_epochs}/{patience})")
            if no_improve_epochs >= patience:
                print("🛑 Early stopping triggered due to no BLEU improvement.")
                break
        else:
            no_improve_epochs = 0  # reset patience if improved
    
    # -------------------
    # Final test BLEU
    # -------------------
    print("\nTraining complete. Evaluating on test set...")
    final_bleu = compute_sample_bleu(model, test_loader, tgt_vocab, max_batches=50)
    print(f"🏁 Final Test BLEU (greedy): {final_bleu:.4f}")


Loading raw data...
Loaded 57426 pairs
Split sizes -> train: 54554 | val: 1723 | test: 1149
Src vocab: 41348 | Tgt vocab: 9977
Starting training...


Epoch 1 [Train]: 100%|██████████| 853/853 [05:21<00:00,  2.66it/s, avg_loss=4.1592]



[Epoch 1] TrainLoss=4.1592 | ValLoss=4.3159 | SampleBLEU=0.0224

🔹 Random Validation Example:
SRC: brakes applying on baby for risk be may there bus crowded in travel not do
REF: startseq ভ ড ব স য ত য় ত করব ন ন এত ব র ক ল গ ন র সময় শ শ র ক ষত হত প র ৷ endseq
PRED: startseq ব র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র র
✅ Saved checkpoint (val loss improved): ./checkpoints/best_loss_seq2seq_bahdanau.pth
✅ Saved checkpoint (BLEU improved): ./checkpoints/best_bleu_seq2seq_bahdanau.pth


Epoch 2 [Train]: 100%|██████████| 853/853 [05:20<00:00,  2.66it/s, avg_loss=3.8419]



[Epoch 2] TrainLoss=3.8419 | ValLoss=4.1837 | SampleBLEU=0.0444

🔹 Random Validation Example:
SRC: town muvattupuzha from away km is nedumbassery at airport international cochin the
REF: startseq ম ভ ত ত প ঝ শহর থ ক ৩৪ ক ল ম ট র দ র অবস থ ত ন দ ম ব স র র ক চ ন আন তর জ ত ক ব ম নবন দর endseq
PRED: startseq ক ন ক র ক র র ক র ক র র ক ল ম ট র ক ম ট র ক ম ট র অবস থ ত endseq
✅ Saved checkpoint (val loss improved): ./checkpoints/best_loss_seq2seq_bahdanau.pth
✅ Saved checkpoint (BLEU improved): ./checkpoints/best_bleu_seq2seq_bahdanau.pth


Epoch 3 [Train]:  52%|█████▏    | 440/853 [02:45<02:35,  2.66it/s, avg_loss=3.6801]


KeyboardInterrupt: 